### data\sec_data_Fulldownloader.py


In [ ]:
#!/usr/bin/env python3
from __future__ import annotations

import argparse
import csv
import os
import dotenv
dotenv.load_dotenv()  # Load .env if present
import re
import time
import zipfile
from pathlib import Path
from typing import Iterable, Iterator

import requests


DATA_ROOT = Path(os.getenv("dataPathGlobal", "data")) / "sec_edgar"
RAW_DIR = DATA_ROOT / "raw"
PROCESSED_DIR = DATA_ROOT / "processed"
LOG_DIR = DATA_ROOT / "logs"

BULK_DIR = RAW_DIR / "bulk"
INDEX_DIR = RAW_DIR / "indexes"
API_DIR = RAW_DIR / "api"
FILING_DIR = RAW_DIR / "filings_txt"

MANIFEST_DIR = PROCESSED_DIR / "manifests"

SEC_USER_AGENT = os.getenv("SEC_USER_AGENT", "").strip()
REQUEST_INTERVAL_SECONDS = 0.15  # < 10 req/s, conservative
TIMEOUT = (20, 180)

BULK_URLS = {
    "submissions_zip": "https://www.sec.gov/Archives/edgar/daily-index/bulkdata/submissions.zip",
    "companyfacts_zip": "https://www.sec.gov/Archives/edgar/daily-index/xbrl/companyfacts.zip",
}

# Your chosen filing scope: D
FORMS_EXACT = {
    "10-K", "10-K/A",
    "10-Q", "10-Q/A",
    "8-K", "8-K/A",
    "DEF 14A", "DEF 14A/A",
    "13D", "13D/A", "SC 13D", "SC 13D/A",
    "13G", "13G/A", "SC 13G", "SC 13G/A",
    "3", "3/A",
    "4", "4/A",
    "5", "5/A",
    "S-1", "S-1/A",
    "S-3", "S-3/A",
    "425",
}
FORM_PREFIXES = ("424B",)

HEADERS = {
    "User-Agent": SEC_USER_AGENT,
    "Accept-Encoding": "gzip, deflate",
}

MASTER_IDX_HEADER_MARKER = "-----"


def require_user_agent() -> None:
    if not SEC_USER_AGENT:
        raise SystemExit(
            "SEC_USER_AGENT is not set.\n"
            "Example:\n"
            'export SEC_USER_AGENT="YourName your_email@example.com"'
        )


def ensure_dirs() -> None:
    for p in [RAW_DIR, PROCESSED_DIR, LOG_DIR, BULK_DIR, INDEX_DIR, API_DIR, FILING_DIR, MANIFEST_DIR]:
        p.mkdir(parents=True, exist_ok=True)


def log(msg: str) -> None:
    print(msg, flush=True)


class SECClient:
    def __init__(self) -> None:
        self.session = requests.Session()
        self.last_request_time = 0.0

    def _throttle(self) -> None:
        elapsed = time.time() - self.last_request_time
        if elapsed < REQUEST_INTERVAL_SECONDS:
            time.sleep(REQUEST_INTERVAL_SECONDS - elapsed)

    def get(self, url: str, stream: bool = False) -> requests.Response:
        last_err = None
        for attempt in range(1, 6):
            try:
                self._throttle()
                resp = self.session.get(url, headers=HEADERS, timeout=TIMEOUT, stream=stream)
                self.last_request_time = time.time()

                if resp.status_code == 200:
                    return resp

                if resp.status_code in (403, 429, 500, 502, 503, 504):
                    wait = min(60, 2 ** attempt)
                    log(f"[retry] {url} -> {resp.status_code}, sleeping {wait}s")
                    time.sleep(wait)
                    continue

                resp.raise_for_status()
            except Exception as exc:  # noqa: BLE001
                last_err = exc
                wait = min(60, 2 ** attempt)
                log(f"[retry-exc] {url} -> {exc}, sleeping {wait}s")
                time.sleep(wait)

        raise RuntimeError(f"Failed to fetch after retries: {url}\nLast error: {last_err}")

    def download_to_file(self, url: str, dest: Path, overwrite: bool = False) -> Path:
        if dest.exists() and not overwrite:
            return dest

        dest.parent.mkdir(parents=True, exist_ok=True)
        tmp = dest.with_suffix(dest.suffix + ".part")

        resp = self.get(url, stream=True)
        with tmp.open("wb") as f:
            for chunk in resp.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
        tmp.replace(dest)
        return dest


def extract_zip(zip_path: Path, extract_to: Path) -> None:
    marker = extract_to / ".extracted.ok"
    if marker.exists():
        return

    extract_to.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_to)
    marker.write_text("ok", encoding="utf-8")


def padded_cik(cik: str | int) -> str:
    return str(cik).strip().zfill(10)


def sanitize_filename(value: str, max_len: int = 120) -> str:
    value = re.sub(r"[^A-Za-z0-9._-]+", "_", value.strip())
    return value[:max_len].strip("_") or "unknown"


def filing_form_wanted(form_type: str) -> bool:
    form_type = form_type.strip().upper()
    if form_type in FORMS_EXACT:
        return True
    return any(form_type.startswith(prefix) for prefix in FORM_PREFIXES)


def quarter_iter(start_year: int, end_year: int) -> Iterator[tuple[int, int]]:
    for year in range(start_year, end_year + 1):
        for qtr in range(1, 5):
            yield year, qtr


def master_idx_url(year: int, qtr: int) -> str:
    return f"https://www.sec.gov/Archives/edgar/full-index/{year}/QTR{qtr}/master.idx"


def master_idx_local_path(year: int, qtr: int) -> Path:
    return INDEX_DIR / "full-index" / str(year) / f"QTR{qtr}" / "master.idx"


def parse_master_idx(path: Path) -> Iterator[dict[str, str]]:
    lines = path.read_text(encoding="latin-1", errors="ignore").splitlines()
    start = None
    for i, line in enumerate(lines):
        if line.startswith(MASTER_IDX_HEADER_MARKER):
            start = i + 1
            break

    if start is None:
        raise ValueError(f"Could not find data header in {path}")

    for line in lines[start:]:
        if not line.strip():
            continue
        parts = line.split("|")
        if len(parts) != 5:
            continue

        cik, company_name, form_type, date_filed, filename = [p.strip() for p in parts]
        yield {
            "cik": cik,
            "company_name": company_name,
            "form_type": form_type,
            "date_filed": date_filed,
            "filename": filename,
            "filing_url": f"https://www.sec.gov/Archives/{filename}",
        }


def accession_from_filename(filename: str) -> str:
    stem = Path(filename).name
    return stem.replace(".txt", "")


def filing_output_path(row: dict[str, str]) -> Path:
    year = row["date_filed"][:4]
    form_type = sanitize_filename(row["form_type"])
    cik = padded_cik(row["cik"])
    accession = accession_from_filename(row["filename"])
    company = sanitize_filename(row["company_name"])
    return FILING_DIR / year / form_type / f"{cik}__{row['date_filed']}__{accession}__{company}.txt"


def save_csv_rows(path: Path, rows: Iterable[dict[str, str]], fieldnames: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


def append_csv_row(path: Path, row: dict[str, str], fieldnames: list[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    file_exists = path.exists()
    with path.open("a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)



def download_bulk_archives(client: SECClient) -> None:
    log("[bulk] downloading SEC bulk archives...")
    submissions_zip = BULK_DIR / "submissions.zip"
    companyfacts_zip = BULK_DIR / "companyfacts.zip"

    client.download_to_file(BULK_URLS["submissions_zip"], submissions_zip)
    client.download_to_file(BULK_URLS["companyfacts_zip"], companyfacts_zip)

    log("[bulk] extracting submissions.zip ...")
    extract_zip(submissions_zip, BULK_DIR / "submissions_extracted")

    log("[bulk] extracting companyfacts.zip ...")
    extract_zip(companyfacts_zip, BULK_DIR / "companyfacts_extracted")

    log("[bulk] done")


def download_single_company_json(client: SECClient, cik: str) -> None:
    cik10 = padded_cik(cik)

    sub_url = f"https://data.sec.gov/submissions/CIK{cik10}.json"
    facts_url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik10}.json"

    sub_dest = API_DIR / "submissions" / f"CIK{cik10}.json"
    facts_dest = API_DIR / "companyfacts" / f"CIK{cik10}.json"

    log(f"[company-json] downloading submissions for CIK {cik10}")
    client.download_to_file(sub_url, sub_dest)

    log(f"[company-json] downloading companyfacts for CIK {cik10}")
    client.download_to_file(facts_url, facts_dest)

    log("[company-json] done")


def build_manifest(client: SECClient, start_year: int, end_year: int) -> Path:
    manifest_path = MANIFEST_DIR / f"filings_manifest_{start_year}_{end_year}.csv"
    fieldnames = [
        "cik",
        "company_name",
        "form_type",
        "date_filed",
        "filename",
        "filing_url",
        "output_path",
    ]

    if manifest_path.exists():
        log(f"[manifest] already exists: {manifest_path}")
        return manifest_path

    log(f"[manifest] building manifest for {start_year}-{end_year} ...")

    with manifest_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for year, qtr in quarter_iter(start_year, end_year):
            idx_url = master_idx_url(year, qtr)
            idx_path = master_idx_local_path(year, qtr)

            log(f"[manifest] index {year} QTR{qtr}")
            client.download_to_file(idx_url, idx_path)

            for row in parse_master_idx(idx_path):
                if not filing_form_wanted(row["form_type"]):
                    continue

                out_path = filing_output_path(row)
                writer.writerow({
                    **row,
                    "output_path": str(out_path),
                })

    log(f"[manifest] written to: {manifest_path}")
    return manifest_path


def download_filings_from_manifest(client: SECClient, manifest_path: Path) -> None:
    if not manifest_path.exists():
        raise FileNotFoundError(f"Manifest not found: {manifest_path}")

    progress_path = LOG_DIR / f"{manifest_path.stem}__download_progress.csv"
    progress_fields = ["filing_url", "output_path", "status", "error"]

    total = 0
    downloaded = 0
    skipped = 0
    failed = 0

    with manifest_path.open("r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)

        for row in reader:
            total += 1
            url = row["filing_url"]
            dest = Path(row["output_path"])

            if dest.exists():
                skipped += 1
                continue

            try:
                client.download_to_file(url, dest)
                downloaded += 1
                append_csv_row(progress_path, {
                    "filing_url": url,
                    "output_path": str(dest),
                    "status": "downloaded",
                    "error": "",
                }, progress_fields)
            except Exception as exc:  # noqa: BLE001
                failed += 1
                append_csv_row(progress_path, {
                    "filing_url": url,
                    "output_path": str(dest),
                    "status": "failed",
                    "error": str(exc),
                }, progress_fields)
                log(f"[filings][failed] {url} -> {exc}")

            if total % 1000 == 0:
                log(
                    f"[filings] processed={total:,} downloaded={downloaded:,} "
                    f"skipped={skipped:,} failed={failed:,}"
                )

    log(
        f"[filings] done. processed={total:,} downloaded={downloaded:,} "
        f"skipped={skipped:,} failed={failed:,}"
    )


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="SEC EDGAR raw downloader: bulk metadata, manifest building, and raw filing text download."
    )
    sub = parser.add_subparsers(dest="command", required=True)

    sub.add_parser("bulk", help="Download and extract submissions.zip and companyfacts.zip")

    company = sub.add_parser("company-json", help="Download single-company submissions and companyfacts JSON")
    company.add_argument("--cik", required=True, help="CIK number, with or without leading zeros")

    manifest = sub.add_parser("manifest", help="Build raw filing manifest from quarterly EDGAR master indexes")
    manifest.add_argument("--start-year", type=int, default=1995)
    manifest.add_argument("--end-year", type=int, default=2024)

    filings = sub.add_parser("filings", help="Download raw filing .txt files from a manifest")
    filings.add_argument("--manifest", required=True, help="Path to manifest CSV")

    all_in_one = sub.add_parser("all", help="Run bulk -> manifest -> filings")
    all_in_one.add_argument("--start-year", type=int, default=1995)
    all_in_one.add_argument("--end-year", type=int, default=2024)

    return parser.parse_args()


def main() -> None:
    require_user_agent()
    ensure_dirs()
    args = parse_args()
    client = SECClient()

    if args.command == "bulk":
        download_bulk_archives(client)

    elif args.command == "company-json":
        download_single_company_json(client, args.cik)

    elif args.command == "manifest":
        build_manifest(client, args.start_year, args.end_year)

    elif args.command == "filings":
        download_filings_from_manifest(client, Path(args.manifest))

    elif args.command == "all":
        download_bulk_archives(client)
        manifest_path = build_manifest(client, args.start_year, args.end_year)
        download_filings_from_manifest(client, manifest_path)

    else:
        raise SystemExit(f"Unknown command: {args.command}")


if __name__ == "__main__":
    main()

# Example Usage:
# python data/sec_data_Fulldownloader.py bulk
# python data/sec_data_Fulldownloader.py filings --manifest data/sec_edgar/processed/filings_manifest_1995_2024.csv
# python data/sec_data_Fulldownloader.py company-json --cik 320193
# python data/sec_data_Fulldownloader.py manifest --start-year 1995 --end-year 2024
# python data/sec_data_Fulldownloader.py all --start-year 1995 --end-year 2024


### data\secCompanyfactsExtraction.py


In [ ]:
from __future__ import annotations

import argparse
import csv
import re
import heapq
import json
import math
import os
import shutil
import sys
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
from typing import Any

try:
    import orjson  # type: ignore
    HAS_ORJSON = True
except ImportError:
    orjson = None
    HAS_ORJSON = False


# ============================================================
# PATHS
# ============================================================

dataPath = Path(os.getenv("dataPathGlobal", "data"))

RAW_INPUT_DIR = dataPath / "sec_edgar" / "raw" / "bulk" / "companyfacts_extracted"
OUT_ROOT = dataPath / "sec_edgar" / "processed" / "companyfacts"

PARTS_DIR = OUT_ROOT / "companyfacts_flat"
SHARD_DIR = OUT_ROOT / "_shards"
TMP_DIR = OUT_ROOT / "_tmp"

INVENTORY_FINAL = OUT_ROOT / "companyfacts_inventory.csv"
ERRORS_FINAL = OUT_ROOT / "companyfacts_errors.csv"
SUMMARY_FINAL = OUT_ROOT / "companyfacts_summary.json"


# ============================================================
# CSV HEADERS
# ============================================================

FACTS_HEADER = [
    "source_json",
    "cik",
    "entity_name",
    "taxonomy",
    "concept",
    "label",
    "description",
    "unit",
    "fy",
    "fp",
    "form",
    "filed",
    "end",
    "frame",
    "accn",
    "val",
]

INVENTORY_HEADER = [
    "source_json",
    "file_size_bytes",
    "cik",
    "entity_name",
    "n_taxonomies",
    "n_concepts",
    "n_units",
    "n_observations",
    "earliest_filed",
    "latest_filed",
    "earliest_end",
    "latest_end",
    "status",
    "error_message",
]

ERRORS_HEADER = [
    "source_json",
    "error_type",
    "error_message",
]


# ============================================================
# HELPERS
# ============================================================
PART_ID_RE = re.compile(r"(?:facts|inventory|errors)_part_(\d{5})\.csv$", re.IGNORECASE)


def load_existing_processed_keys(inventory_csv: Path) -> tuple[set[str], set[str]]:
    """
    Returns:
      processed_relpaths: source_json values from existing inventory
      processed_basenames: file basenames from source_json

    We keep both because if the user copied files back with slightly different
    folder nesting, basename-based skipping still works for SEC companyfacts,
    whose filenames are effectively unique per CIK.
    """
    processed_relpaths: set[str] = set()
    processed_basenames: set[str] = set()

    if not inventory_csv.exists():
        return processed_relpaths, processed_basenames

    with inventory_csv.open("r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        if not reader.fieldnames or "source_json" not in reader.fieldnames:
            raise RuntimeError(
                f"Existing inventory file does not have expected 'source_json' column: {inventory_csv}"
            )

        for row in reader:
            src = (row.get("source_json") or "").strip()
            if not src:
                continue
            src = src.replace("\\", "/")
            processed_relpaths.add(src)
            processed_basenames.add(Path(src).name)

    return processed_relpaths, processed_basenames


def next_available_part_id(parts_dir: Path) -> int:
    """
    Finds the next shard/part number by scanning existing part files.
    """
    max_id = 0
    if parts_dir.exists():
        for p in parts_dir.glob("*_part_*.csv"):
            m = PART_ID_RE.match(p.name)
            if m:
                max_id = max(max_id, int(m.group(1)))
    return max_id + 1


def load_previous_summary(summary_path: Path) -> dict[str, Any]:
    if not summary_path.exists():
        return {}
    try:
        with summary_path.open("r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return {}

def human_bytes(n: int | float) -> str:
    n = float(n)
    units = ["B", "KB", "MB", "GB", "TB"]
    for unit in units:
        if n < 1024.0 or unit == units[-1]:
            return f"{n:.2f} {unit}"
        n /= 1024.0
    return f"{n:.2f} B"


def now_ts() -> float:
    return time.time()


def fmt_elapsed(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.1f}s"
    minutes = seconds / 60.0
    if minutes < 60:
        return f"{minutes:.1f}m"
    hours = minutes / 60.0
    return f"{hours:.2f}h"


def json_loads_bytes(raw: bytes) -> Any:
    if HAS_ORJSON:
        return orjson.loads(raw)
    return json.loads(raw.decode("utf-8"))


def ensure_clean_output_root(overwrite: bool) -> None:
    OUT_ROOT.mkdir(parents=True, exist_ok=True)

    if overwrite:
        for p in [PARTS_DIR, SHARD_DIR, TMP_DIR, INVENTORY_FINAL, ERRORS_FINAL, SUMMARY_FINAL]:
            if p.is_dir():
                shutil.rmtree(p, ignore_errors=True)
            elif p.exists():
                p.unlink()

    PARTS_DIR.mkdir(parents=True, exist_ok=True)
    SHARD_DIR.mkdir(parents=True, exist_ok=True)
    TMP_DIR.mkdir(parents=True, exist_ok=True)


def discover_json_files(input_dir: Path) -> list[Path]:
    if not input_dir.exists():
        raise FileNotFoundError(f"Input directory not found: {input_dir}")
    files = sorted(input_dir.rglob("*.json"))
    if not files:
        raise FileNotFoundError(f"No JSON files found under: {input_dir}")
    return files


def greedy_partition(files: list[Path], shard_count: int) -> list[list[Path]]:
    """
    Greedy size-balanced partitioning.
    Sort biggest files first, always assign next file to currently lightest shard.
    """
    shard_count = max(1, min(shard_count, len(files)))
    shards: list[list[Path]] = [[] for _ in range(shard_count)]
    heap: list[tuple[int, int]] = [(0, i) for i in range(shard_count)]
    heapq.heapify(heap)

    file_sizes = []
    for p in files:
        try:
            size = p.stat().st_size
        except Exception:
            size = 0
        file_sizes.append((size, p))

    file_sizes.sort(key=lambda x: x[0], reverse=True)

    for size, path in file_sizes:
        total_size, shard_idx = heapq.heappop(heap)
        shards[shard_idx].append(path)
        heapq.heappush(heap, (total_size + size, shard_idx))

    return [s for s in shards if s]


def write_csv_header(path: Path, header: list[str]) -> None:
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(header)


def copy_fileobj_in_chunks(src_fh, dst_fh, chunk_size: int = 16 * 1024 * 1024) -> int:
    copied = 0
    while True:
        chunk = src_fh.read(chunk_size)
        if not chunk:
            break
        dst_fh.write(chunk)
        copied += len(chunk)
    return copied


def merge_csv_parts(
    part_paths: list[Path],
    out_path: Path,
    progress_prefix: str,
    progress_every_mb: int = 512,
) -> None:
    if not part_paths:
        raise ValueError(f"No part files provided for merge into {out_path}")

    total_bytes = sum(p.stat().st_size for p in part_paths if p.exists())
    done_bytes = 0
    next_report = progress_every_mb * 1024 * 1024
    start = now_ts()

    tmp_out = out_path.with_suffix(out_path.suffix + ".part")
    if tmp_out.exists():
        tmp_out.unlink()

    with tmp_out.open("wb") as out_fh:
        for idx, part in enumerate(part_paths):
            if not part.exists():
                continue

            with part.open("rb") as in_fh:
                if idx > 0:
                    # Skip header line for all but first file
                    in_fh.readline()
                copied = copy_fileobj_in_chunks(in_fh, out_fh)
                done_bytes += copied

                if done_bytes >= next_report or done_bytes == total_bytes:
                    elapsed = max(now_ts() - start, 1e-9)
                    speed = done_bytes / elapsed
                    pct = (done_bytes / total_bytes * 100.0) if total_bytes else 100.0
                    eta = (total_bytes - done_bytes) / speed if speed > 0 else 0.0
                    print(
                        f"{progress_prefix} | {pct:6.2f}% | "
                        f"{human_bytes(done_bytes)} / {human_bytes(total_bytes)} | "
                        f"{human_bytes(speed)}/s | ETA {fmt_elapsed(eta)}",
                        flush=True,
                    )
                    next_report += progress_every_mb * 1024 * 1024

    tmp_out.replace(out_path)


def safe_min_date(current: str, new_val: str) -> str:
    if not new_val:
        return current
    if not current:
        return new_val
    return min(current, new_val)


def safe_max_date(current: str, new_val: str) -> str:
    if not new_val:
        return current
    if not current:
        return new_val
    return max(current, new_val)


# ============================================================
# WORKER
# ============================================================

def process_shard(
    shard_id: int,
    files: list[str],
    input_root_str: str,
    shard_dir_str: str,
) -> dict[str, Any]:
    """
    Each worker processes a list of JSON files and writes:
      - one facts CSV part
      - one inventory CSV shard
      - one errors CSV shard
    """
    input_root = Path(input_root_str)
    shard_dir = Path(shard_dir_str)

    facts_part_path = shard_dir / f"facts_part_{shard_id:05d}.csv"
    inventory_part_path = shard_dir / f"inventory_part_{shard_id:05d}.csv"
    errors_part_path = shard_dir / f"errors_part_{shard_id:05d}.csv"

    write_csv_header(facts_part_path, FACTS_HEADER)
    write_csv_header(inventory_part_path, INVENTORY_HEADER)
    write_csv_header(errors_part_path, ERRORS_HEADER)

    facts_rows = 0
    inventory_rows = 0
    error_rows = 0
    ok_files = 0
    failed_files = 0

    with (
        facts_part_path.open("a", newline="", encoding="utf-8") as facts_fh,
        inventory_part_path.open("a", newline="", encoding="utf-8") as inv_fh,
        errors_part_path.open("a", newline="", encoding="utf-8") as err_fh,
    ):
        facts_writer = csv.writer(facts_fh)
        inv_writer = csv.writer(inv_fh)
        err_writer = csv.writer(err_fh)

        for file_str in files:
            path = Path(file_str)

            try:
                rel_path = str(path.relative_to(input_root)).replace("\\", "/")
            except Exception:
                rel_path = str(path).replace("\\", "/")

            file_size = 0
            try:
                file_size = path.stat().st_size
            except Exception:
                pass

            try:
                raw = path.read_bytes()
                obj = json_loads_bytes(raw)

                cik = obj.get("cik", "")
                entity_name = obj.get("entityName", "")
                facts = obj.get("facts", {}) or {}

                n_taxonomies = 0
                n_concepts = 0
                n_units = 0
                n_observations = 0
                earliest_filed = ""
                latest_filed = ""
                earliest_end = ""
                latest_end = ""

                if not isinstance(facts, dict) or not facts:
                    inv_writer.writerow([
                        rel_path,
                        file_size,
                        cik,
                        entity_name,
                        0,
                        0,
                        0,
                        0,
                        "",
                        "",
                        "",
                        "",
                        "no_facts",
                        "",
                    ])
                    inventory_rows += 1
                    ok_files += 1
                    continue

                for taxonomy, concept_map in facts.items():
                    if not isinstance(concept_map, dict):
                        continue

                    n_taxonomies += 1

                    for concept, concept_payload in concept_map.items():
                        if not isinstance(concept_payload, dict):
                            continue

                        n_concepts += 1

                        label = concept_payload.get("label", "")
                        description = concept_payload.get("description", "")
                        units = concept_payload.get("units", {}) or {}

                        if not isinstance(units, dict):
                            continue

                        for unit, obs_list in units.items():
                            n_units += 1

                            if not isinstance(obs_list, list):
                                continue

                            for obs in obs_list:
                                if not isinstance(obs, dict):
                                    continue

                                fy = obs.get("fy", "")
                                fp = obs.get("fp", "")
                                form = obs.get("form", "")
                                filed = obs.get("filed", "")
                                end = obs.get("end", "")
                                frame = obs.get("frame", "")
                                accn = obs.get("accn", "")
                                val = obs.get("val", "")

                                earliest_filed = safe_min_date(earliest_filed, filed)
                                latest_filed = safe_max_date(latest_filed, filed)
                                earliest_end = safe_min_date(earliest_end, end)
                                latest_end = safe_max_date(latest_end, end)

                                facts_writer.writerow([
                                    rel_path,
                                    cik,
                                    entity_name,
                                    taxonomy,
                                    concept,
                                    label,
                                    description,
                                    unit,
                                    fy,
                                    fp,
                                    form,
                                    filed,
                                    end,
                                    frame,
                                    accn,
                                    val,
                                ])
                                facts_rows += 1
                                n_observations += 1

                inv_writer.writerow([
                    rel_path,
                    file_size,
                    cik,
                    entity_name,
                    n_taxonomies,
                    n_concepts,
                    n_units,
                    n_observations,
                    earliest_filed,
                    latest_filed,
                    earliest_end,
                    latest_end,
                    "ok",
                    "",
                ])
                inventory_rows += 1
                ok_files += 1

            except Exception as exc:
                failed_files += 1
                error_rows += 1

                err_writer.writerow([
                    rel_path,
                    type(exc).__name__,
                    str(exc),
                ])

                inv_writer.writerow([
                    rel_path,
                    file_size,
                    "",
                    "",
                    0,
                    0,
                    0,
                    0,
                    "",
                    "",
                    "",
                    "",
                    "error",
                    str(exc),
                ])
                inventory_rows += 1

    return {
        "shard_id": shard_id,
        "facts_part": str(facts_part_path),
        "inventory_part": str(inventory_part_path),
        "errors_part": str(errors_part_path),
        "input_files": len(files),
        "ok_files": ok_files,
        "failed_files": failed_files,
        "facts_rows": facts_rows,
        "inventory_rows": inventory_rows,
        "error_rows": error_rows,
    }


# ============================================================
# MAIN PIPELINE
# ============================================================

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Inventory + flatten SEC companyfacts JSON files into partitioned CSVs."
    )
    parser.add_argument(
        "--resume",
        action="store_true",
        help="Incremental mode: skip files already recorded in existing companyfacts_inventory.csv and append only new files.",
    )
    parser.add_argument(
        "--input-dir",
        type=str,
        default=str(RAW_INPUT_DIR),
        help="Directory containing extracted SEC companyfacts JSON files.",
    )
    parser.add_argument(
        "--workers",
        type=int,
        default=0,
        help="Number of worker processes. 0 = auto.",
    )
    parser.add_argument(
        "--shard-multiplier",
        type=int,
        default=4,
        help="Number of shards ~= workers * shard_multiplier. More shards improves load balance.",
    )
    parser.add_argument(
        "--overwrite",
        action="store_true",
        help="Delete previous processed outputs before running.",
    )
    return parser.parse_args()


def choose_worker_count(requested: int) -> int:
    if requested and requested > 0:
        return requested
    cpu = os.cpu_count() or 4
    if cpu <= 2:
        return 1
    return max(1, cpu - 1)


def main() -> None:
    args = parse_args()

    input_dir = Path(args.input_dir)
    workers = choose_worker_count(args.workers)

    ensure_clean_output_root(args.overwrite)

    print(f"Input directory : {input_dir}", flush=True)
    print(f"Output root     : {OUT_ROOT}", flush=True)
    print(f"JSON parser     : {'orjson' if HAS_ORJSON else 'stdlib json'}", flush=True)
    print(f"Workers         : {workers}", flush=True)
    print(f"Resume mode     : {args.resume}", flush=True)

    overall_start = now_ts()

    # --------------------------------------------------------
    # Discover files
    # --------------------------------------------------------
    discover_start = now_ts()
    all_files = discover_json_files(input_dir)
    all_input_bytes = sum(p.stat().st_size for p in all_files)
    discover_elapsed = now_ts() - discover_start

    print(
        f"Discovered {len(all_files):,} companyfacts JSON files "
        f"({human_bytes(all_input_bytes)}) in {fmt_elapsed(discover_elapsed)}",
        flush=True,
    )

    # --------------------------------------------------------
    # Incremental resume filtering
    # --------------------------------------------------------
    previous_summary = load_previous_summary(SUMMARY_FINAL)

    processed_relpaths: set[str] = set()
    processed_basenames: set[str] = set()
    files = all_files
    skipped_existing = 0

    if args.resume and INVENTORY_FINAL.exists():
        processed_relpaths, processed_basenames = load_existing_processed_keys(INVENTORY_FINAL)

        filtered: list[Path] = []
        for p in all_files:
            rel = str(p.relative_to(input_dir)).replace("\\", "/")
            base = p.name

            if rel in processed_relpaths or base in processed_basenames:
                skipped_existing += 1
                continue

            filtered.append(p)

        files = filtered

        print(
            f"Existing processed files detected: {len(processed_basenames):,} "
            f"(skipping {skipped_existing:,} already processed files)",
            flush=True,
        )

    if not files:
        print("No new companyfacts JSON files to process. Nothing to do.", flush=True)
        return

    new_input_bytes = sum(p.stat().st_size for p in files)
    print(
        f"New files to process: {len(files):,} "
        f"({human_bytes(new_input_bytes)})",
        flush=True,
    )

    # --------------------------------------------------------
    # Partition into shards
    # --------------------------------------------------------
    shard_count = max(1, min(len(files), workers * max(1, args.shard_multiplier)))
    shards = greedy_partition(files, shard_count)

    first_part_id = next_available_part_id(PARTS_DIR)

    print(f"Shard count       : {len(shards)}", flush=True)
    print(f"Starting part id  : {first_part_id:05d}", flush=True)
    print(f"Parts directory   : {PARTS_DIR}", flush=True)

    # --------------------------------------------------------
    # Process shards in parallel
    # --------------------------------------------------------
    futures = []
    shard_start = now_ts()

    total_files_done = 0
    total_ok_files = 0
    total_failed_files = 0
    total_fact_rows = 0
    total_inventory_rows = 0
    total_error_rows = 0

    with ProcessPoolExecutor(max_workers=workers) as executor:
        for offset, shard_files in enumerate(shards):
            shard_id = first_part_id + offset
            futures.append(
                executor.submit(
                    process_shard,
                    shard_id,
                    [str(p) for p in shard_files],
                    str(input_dir),
                    str(PARTS_DIR),
                )
            )

        completed = 0
        total_shards = len(futures)

        for fut in as_completed(futures):
            result = fut.result()
            completed += 1

            total_files_done += result["input_files"]
            total_ok_files += result["ok_files"]
            total_failed_files += result["failed_files"]
            total_fact_rows += result["facts_rows"]
            total_inventory_rows += result["inventory_rows"]
            total_error_rows += result["error_rows"]

            elapsed = max(now_ts() - shard_start, 1e-9)
            files_per_sec = total_files_done / elapsed
            rows_per_sec = total_fact_rows / elapsed if total_fact_rows else 0.0

            print(
                f"[process] shards {completed}/{total_shards} | "
                f"new files {total_files_done:,}/{len(files):,} | "
                f"ok {total_ok_files:,} | failed {total_failed_files:,} | "
                f"facts rows {total_fact_rows:,} | "
                f"{files_per_sec:,.1f} files/s | {rows_per_sec:,.1f} rows/s",
                flush=True,
            )

    process_elapsed = now_ts() - shard_start

    # --------------------------------------------------------
    # Rebuild final inventory and errors from all shard parts
    # --------------------------------------------------------
    inventory_parts = sorted(PARTS_DIR.glob("inventory_part_*.csv"))
    error_parts = sorted(PARTS_DIR.glob("errors_part_*.csv"))

    print("Rebuilding cumulative inventory CSV...", flush=True)
    merge_csv_parts(inventory_parts, INVENTORY_FINAL, "[merge inventory]")

    print("Rebuilding cumulative errors CSV...", flush=True)
    merge_csv_parts(error_parts, ERRORS_FINAL, "[merge errors]")

    # --------------------------------------------------------
    # Facts parts remain partitioned output
    # --------------------------------------------------------
    facts_parts = sorted(PARTS_DIR.glob("facts_part_*.csv"))

    previous_total_fact_rows = int(previous_summary.get("total_fact_rows", 0) or 0)
    previous_total_ok_files = int(previous_summary.get("total_ok_files", 0) or 0)
    previous_total_failed_files = int(previous_summary.get("total_failed_files", 0) or 0)

    cumulative_total_fact_rows = previous_total_fact_rows + total_fact_rows
    cumulative_total_ok_files = previous_total_ok_files + total_ok_files
    cumulative_total_failed_files = previous_total_failed_files + total_failed_files

    summary = {
        "input_dir": str(input_dir),
        "output_root": str(OUT_ROOT),
        "used_orjson": HAS_ORJSON,
        "workers": workers,
        "resume_mode": args.resume,
        "shard_count_this_run": len(shards),
        "starting_part_id_this_run": first_part_id,
        "json_files_discovered_total": len(all_files),
        "json_files_processed_this_run": len(files),
        "json_files_skipped_as_existing": skipped_existing,
        "total_input_bytes_all_visible": all_input_bytes,
        "total_input_bytes_processed_this_run": new_input_bytes,
        "total_ok_files_this_run": total_ok_files,
        "total_failed_files_this_run": total_failed_files,
        "total_inventory_rows_this_run": total_inventory_rows,
        "total_error_rows_this_run": total_error_rows,
        "total_fact_rows_this_run": total_fact_rows,
        "total_ok_files": cumulative_total_ok_files,
        "total_failed_files": cumulative_total_failed_files,
        "total_fact_rows": cumulative_total_fact_rows,
        "facts_parts": [str(p) for p in facts_parts],
        "inventory_csv": str(INVENTORY_FINAL),
        "errors_csv": str(ERRORS_FINAL),
        "timing": {
            "processing_seconds_this_run": process_elapsed,
            "total_seconds_this_run": now_ts() - overall_start,
        },
    }

    with SUMMARY_FINAL.open("w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    total_elapsed = now_ts() - overall_start

    print("\n=== COMPANYFACTS PIPELINE COMPLETE ===", flush=True)
    print(f"Inventory CSV   : {INVENTORY_FINAL}", flush=True)
    print(f"Errors CSV      : {ERRORS_FINAL}", flush=True)
    print(f"Facts parts     : {len(facts_parts)} file(s) in {PARTS_DIR}", flush=True)
    print(f"Summary JSON    : {SUMMARY_FINAL}", flush=True)
    print(f"All visible JSON : {len(all_files):,}", flush=True)
    print(f"Processed this run: {len(files):,}", flush=True)
    print(f"Skipped existing : {skipped_existing:,}", flush=True)
    print(f"New fact rows    : {total_fact_rows:,}", flush=True)
    print(f"Cumulative rows  : {cumulative_total_fact_rows:,}", flush=True)
    print(f"Elapsed          : {fmt_elapsed(total_elapsed)}", flush=True)

if __name__ == "__main__":
    # Required for safe multiprocessing on Windows
    main()

### data\secSubmissionsExtraction.py

In [ ]:
from __future__ import annotations

import argparse
import csv
import heapq
import json
import os
import re
import shutil
import sys
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
from typing import Any

try:
    import orjson  # type: ignore
    HAS_ORJSON = True
except ImportError:
    orjson = None
    HAS_ORJSON = False


# ============================================================
# PATHS
# ============================================================

dataPath = Path(os.getenv("dataPathGlobal", "data"))

RAW_INPUT_DIR = dataPath / "sec_edgar" / "raw" / "bulk" / "submissions_extracted"
OUT_ROOT = dataPath / "sec_edgar" / "processed" / "submissions"

PARTS_DIR = OUT_ROOT / "submissions_flat"
TMP_DIR = OUT_ROOT / "_tmp"

INVENTORY_FINAL = OUT_ROOT / "submissions_inventory.csv"
ERRORS_FINAL = OUT_ROOT / "submissions_errors.csv"
SUMMARY_FINAL = OUT_ROOT / "submissions_summary.json"


# ============================================================
# CSV HEADERS
# ============================================================

INVENTORY_HEADER = [
    "source_json",
    "file_size_bytes",
    "cik",
    "entity_name",
    "entity_type",
    "n_tickers",
    "n_exchanges",
    "n_former_names",
    "n_recent_filings",
    "n_file_refs",
    "earliest_filing_date",
    "latest_filing_date",
    "status",
    "error_message",
]

ERRORS_HEADER = [
    "source_json",
    "error_type",
    "error_message",
]

ENTITIES_HEADER = [
    "source_json",
    "cik",
    "entityType",
    "sic",
    "sicDescription",
    "ownerOrg",
    "insiderTransactionForOwnerExists",
    "insiderTransactionForIssuerExists",
    "name",
    "tickers",
    "exchanges",
    "ein",
    "description",
    "website",
    "investorWebsite",
    "category",
    "fiscalYearEnd",
    "stateOfIncorporation",
    "stateOfIncorporationDescription",
    "phone",
    "flags",
    "mailing_street1",
    "mailing_street2",
    "mailing_city",
    "mailing_stateOrCountry",
    "mailing_stateOrCountryDescription",
    "mailing_zipCode",
    "business_street1",
    "business_street2",
    "business_city",
    "business_stateOrCountry",
    "business_stateOrCountryDescription",
    "business_zipCode",
]

RECENT_FILINGS_HEADER = [
    "source_json",
    "cik",
    "accessionNumber",
    "filingDate",
    "reportDate",
    "acceptanceDateTime",
    "act",
    "form",
    "fileNumber",
    "filmNumber",
    "items",
    "size",
    "isXBRL",
    "isInlineXBRL",
    "primaryDocument",
    "primaryDocDescription",
]

FILING_FILES_HEADER = [
    "source_json",
    "cik",
    "name",
    "filingCount",
    "filingFrom",
    "filingTo",
]

FORMER_NAMES_HEADER = [
    "source_json",
    "cik",
    "name",
    "from_date",
    "to_date",
]


# ============================================================
# HELPERS
# ============================================================

PART_ID_RE = re.compile(
    r"(?:inventory|errors|entities|recent_filings|filing_files|former_names)_part_(\d{5})\.csv$",
    re.IGNORECASE,
)


class LowDiskSpaceError(RuntimeError):
    pass


def human_bytes(n: int | float) -> str:
    n = float(n)
    units = ["B", "KB", "MB", "GB", "TB"]
    for unit in units:
        if n < 1024.0 or unit == units[-1]:
            return f"{n:.2f} {unit}"
        n /= 1024.0
    return f"{n:.2f} B"


def now_ts() -> float:
    return time.time()


def fmt_elapsed(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.1f}s"
    minutes = seconds / 60.0
    if minutes < 60:
        return f"{minutes:.1f}m"
    hours = minutes / 60.0
    return f"{hours:.2f}h"


def json_loads_bytes(raw: bytes) -> Any:
    if HAS_ORJSON:
        return orjson.loads(raw)
    return json.loads(raw.decode("utf-8"))


def write_csv_header(path: Path, header: list[str]) -> None:
    with path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(header)


def copy_fileobj_in_chunks(src_fh, dst_fh, chunk_size: int = 16 * 1024 * 1024) -> int:
    copied = 0
    while True:
        chunk = src_fh.read(chunk_size)
        if not chunk:
            break
        dst_fh.write(chunk)
        copied += len(chunk)
    return copied


def merge_csv_parts(
    part_paths: list[Path],
    out_path: Path,
    progress_prefix: str,
    progress_every_mb: int = 512,
) -> None:
    if not part_paths:
        raise ValueError(f"No part files provided for merge into {out_path}")

    total_bytes = sum(p.stat().st_size for p in part_paths if p.exists())
    done_bytes = 0
    next_report = progress_every_mb * 1024 * 1024
    start = now_ts()

    tmp_out = out_path.with_suffix(out_path.suffix + ".part")
    if tmp_out.exists():
        tmp_out.unlink()

    with tmp_out.open("wb") as out_fh:
        for idx, part in enumerate(part_paths):
            if not part.exists():
                continue

            with part.open("rb") as in_fh:
                if idx > 0:
                    in_fh.readline()  # skip header in later parts
                copied = copy_fileobj_in_chunks(in_fh, out_fh)
                done_bytes += copied

                if done_bytes >= next_report or done_bytes == total_bytes:
                    elapsed = max(now_ts() - start, 1e-9)
                    speed = done_bytes / elapsed
                    pct = (done_bytes / total_bytes * 100.0) if total_bytes else 100.0
                    eta = (total_bytes - done_bytes) / speed if speed > 0 else 0.0
                    print(
                        f"{progress_prefix} | {pct:6.2f}% | "
                        f"{human_bytes(done_bytes)} / {human_bytes(total_bytes)} | "
                        f"{human_bytes(speed)}/s | ETA {fmt_elapsed(eta)}",
                        flush=True,
                    )
                    next_report += progress_every_mb * 1024 * 1024

    tmp_out.replace(out_path)


def safe_min_date(current: str, new_val: str) -> str:
    if not new_val:
        return current
    if not current:
        return new_val
    return min(current, new_val)


def safe_max_date(current: str, new_val: str) -> str:
    if not new_val:
        return current
    if not current:
        return new_val
    return max(current, new_val)


def list_to_pipe_str(value: Any) -> str:
    if isinstance(value, list):
        return "|".join("" if v is None else str(v) for v in value)
    if value is None:
        return ""
    return str(value)


def boolish_to_str(value: Any) -> str:
    if value is None:
        return ""
    return str(value)


def get_address_field(address_obj: dict[str, Any] | None, key: str) -> str:
    if not isinstance(address_obj, dict):
        return ""
    v = address_obj.get(key, "")
    return "" if v is None else str(v)


def choose_worker_count(requested: int) -> int:
    if requested and requested > 0:
        return requested
    cpu = os.cpu_count() or 4
    if cpu <= 2:
        return 1
    return max(1, cpu - 1)


def ensure_dirs(overwrite: bool) -> None:
    OUT_ROOT.mkdir(parents=True, exist_ok=True)

    if overwrite:
        for p in [PARTS_DIR, TMP_DIR, INVENTORY_FINAL, ERRORS_FINAL, SUMMARY_FINAL]:
            if p.is_dir():
                shutil.rmtree(p, ignore_errors=True)
            elif p.exists():
                p.unlink()

    PARTS_DIR.mkdir(parents=True, exist_ok=True)
    TMP_DIR.mkdir(parents=True, exist_ok=True)


def discover_json_files(input_dir: Path) -> list[Path]:
    if not input_dir.exists():
        raise FileNotFoundError(f"Input directory not found: {input_dir}")
    files = sorted(input_dir.rglob("*.json"))
    if not files:
        raise FileNotFoundError(f"No JSON files found under: {input_dir}")
    return files


def greedy_partition(files: list[Path], shard_count: int) -> list[list[Path]]:
    shard_count = max(1, min(shard_count, len(files)))
    shards: list[list[Path]] = [[] for _ in range(shard_count)]
    heap: list[tuple[int, int]] = [(0, i) for i in range(shard_count)]
    heapq.heapify(heap)

    file_sizes = []
    for p in files:
        try:
            size = p.stat().st_size
        except Exception:
            size = 0
        file_sizes.append((size, p))

    file_sizes.sort(key=lambda x: x[0], reverse=True)

    for size, path in file_sizes:
        total_size, shard_idx = heapq.heappop(heap)
        shards[shard_idx].append(path)
        heapq.heappush(heap, (total_size + size, shard_idx))

    return [s for s in shards if s]


def next_available_part_id(parts_dir: Path) -> int:
    max_id = 0
    if parts_dir.exists():
        for p in parts_dir.glob("*_part_*.csv"):
            m = PART_ID_RE.match(p.name)
            if m:
                max_id = max(max_id, int(m.group(1)))
    return max_id + 1


def load_previous_summary(summary_path: Path) -> dict[str, Any]:
    if not summary_path.exists():
        return {}
    try:
        with summary_path.open("r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return {}


def iter_existing_inventory_sources() -> list[Path]:
    paths: list[Path] = []
    if INVENTORY_FINAL.exists():
        paths.append(INVENTORY_FINAL)
    paths.extend(sorted(PARTS_DIR.glob("inventory_part_*.csv")))
    return paths


def load_existing_processed_keys() -> tuple[set[str], set[str]]:
    """
    Returns:
      processed_relpaths
      processed_basenames

    We load from both the final merged inventory and any shard inventory parts.
    This makes resume robust even if the previous run terminated before final merge.
    """
    processed_relpaths: set[str] = set()
    processed_basenames: set[str] = set()

    for inventory_csv in iter_existing_inventory_sources():
        try:
            with inventory_csv.open("r", newline="", encoding="utf-8") as f:
                reader = csv.DictReader(f)
                if not reader.fieldnames or "source_json" not in reader.fieldnames:
                    continue

                for row in reader:
                    src = (row.get("source_json") or "").strip()
                    if not src:
                        continue
                    src = src.replace("\\", "/")
                    processed_relpaths.add(src)
                    processed_basenames.add(Path(src).name)
        except Exception:
            # corrupted partial inventory shards should not kill resume
            continue

    return processed_relpaths, processed_basenames


def free_bytes(path: Path) -> int:
    usage = shutil.disk_usage(path)
    return int(usage.free)


def check_disk_or_raise(base_path: Path, min_free_bytes: int, context: str) -> None:
    available = free_bytes(base_path)
    if available < min_free_bytes:
        raise LowDiskSpaceError(
            f"Low disk space during {context}. "
            f"Available={human_bytes(available)}, required minimum={human_bytes(min_free_bytes)}"
        )


def safe_list_len(value: Any) -> int:
    return len(value) if isinstance(value, list) else 0


def safe_list_item(lst: Any, idx: int) -> str:
    if not isinstance(lst, list):
        return ""
    if idx >= len(lst):
        return ""
    val = lst[idx]
    return "" if val is None else str(val)


# ============================================================
# WORKER
# ============================================================

def process_shard(
    shard_id: int,
    files: list[str],
    input_root_str: str,
    parts_dir_str: str,
    min_free_bytes: int,
    disk_check_every_files: int,
) -> dict[str, Any]:
    input_root = Path(input_root_str)
    parts_dir = Path(parts_dir_str)

    inventory_part_path = parts_dir / f"inventory_part_{shard_id:05d}.csv"
    errors_part_path = parts_dir / f"errors_part_{shard_id:05d}.csv"
    entities_part_path = parts_dir / f"entities_part_{shard_id:05d}.csv"
    recent_filings_part_path = parts_dir / f"recent_filings_part_{shard_id:05d}.csv"
    filing_files_part_path = parts_dir / f"filing_files_part_{shard_id:05d}.csv"
    former_names_part_path = parts_dir / f"former_names_part_{shard_id:05d}.csv"

    write_csv_header(inventory_part_path, INVENTORY_HEADER)
    write_csv_header(errors_part_path, ERRORS_HEADER)
    write_csv_header(entities_part_path, ENTITIES_HEADER)
    write_csv_header(recent_filings_part_path, RECENT_FILINGS_HEADER)
    write_csv_header(filing_files_part_path, FILING_FILES_HEADER)
    write_csv_header(former_names_part_path, FORMER_NAMES_HEADER)

    inventory_rows = 0
    error_rows = 0
    entity_rows = 0
    recent_rows = 0
    filing_file_rows = 0
    former_name_rows = 0
    ok_files = 0
    failed_files = 0

    with (
        inventory_part_path.open("a", newline="", encoding="utf-8") as inv_fh,
        errors_part_path.open("a", newline="", encoding="utf-8") as err_fh,
        entities_part_path.open("a", newline="", encoding="utf-8") as ent_fh,
        recent_filings_part_path.open("a", newline="", encoding="utf-8") as recent_fh,
        filing_files_part_path.open("a", newline="", encoding="utf-8") as file_fh,
        former_names_part_path.open("a", newline="", encoding="utf-8") as former_fh,
    ):
        inv_writer = csv.writer(inv_fh)
        err_writer = csv.writer(err_fh)
        ent_writer = csv.writer(ent_fh)
        recent_writer = csv.writer(recent_fh)
        file_writer = csv.writer(file_fh)
        former_writer = csv.writer(former_fh)

        for file_idx, file_str in enumerate(files, start=1):
            if file_idx % max(1, disk_check_every_files) == 0:
                check_disk_or_raise(parts_dir, min_free_bytes, f"processing shard {shard_id}")

            path = Path(file_str)
            try:
                rel_path = str(path.relative_to(input_root)).replace("\\", "/")
            except Exception:
                rel_path = str(path).replace("\\", "/")

            file_size = 0
            try:
                file_size = path.stat().st_size
            except Exception:
                pass

            try:
                raw = path.read_bytes()
                obj = json_loads_bytes(raw)

                cik = "" if obj.get("cik") is None else str(obj.get("cik"))
                entity_name = "" if obj.get("name") is None else str(obj.get("name"))
                entity_type = "" if obj.get("entityType") is None else str(obj.get("entityType"))

                tickers = obj.get("tickers", [])
                exchanges = obj.get("exchanges", [])
                former_names = obj.get("formerNames", [])
                filings = obj.get("filings", {}) or {}
                recent = filings.get("recent", {}) or {}
                filing_files = filings.get("files", []) or []

                n_tickers = safe_list_len(tickers)
                n_exchanges = safe_list_len(exchanges)
                n_former_names = safe_list_len(former_names)
                n_file_refs = safe_list_len(filing_files)

                # ------------------------------------------------
                # entity metadata row
                # ------------------------------------------------
                mailing = obj.get("addresses", {}).get("mailing", {}) if isinstance(obj.get("addresses"), dict) else {}
                business = obj.get("addresses", {}).get("business", {}) if isinstance(obj.get("addresses"), dict) else {}

                ent_writer.writerow([
                    rel_path,
                    cik,
                    entity_type,
                    "" if obj.get("sic") is None else str(obj.get("sic")),
                    "" if obj.get("sicDescription") is None else str(obj.get("sicDescription")),
                    "" if obj.get("ownerOrg") is None else str(obj.get("ownerOrg")),
                    boolish_to_str(obj.get("insiderTransactionForOwnerExists")),
                    boolish_to_str(obj.get("insiderTransactionForIssuerExists")),
                    entity_name,
                    list_to_pipe_str(tickers),
                    list_to_pipe_str(exchanges),
                    "" if obj.get("ein") is None else str(obj.get("ein")),
                    "" if obj.get("description") is None else str(obj.get("description")),
                    "" if obj.get("website") is None else str(obj.get("website")),
                    "" if obj.get("investorWebsite") is None else str(obj.get("investorWebsite")),
                    "" if obj.get("category") is None else str(obj.get("category")),
                    "" if obj.get("fiscalYearEnd") is None else str(obj.get("fiscalYearEnd")),
                    "" if obj.get("stateOfIncorporation") is None else str(obj.get("stateOfIncorporation")),
                    "" if obj.get("stateOfIncorporationDescription") is None else str(obj.get("stateOfIncorporationDescription")),
                    "" if obj.get("phone") is None else str(obj.get("phone")),
                    "" if obj.get("flags") is None else str(obj.get("flags")),
                    get_address_field(mailing, "street1"),
                    get_address_field(mailing, "street2"),
                    get_address_field(mailing, "city"),
                    get_address_field(mailing, "stateOrCountry"),
                    get_address_field(mailing, "stateOrCountryDescription"),
                    get_address_field(mailing, "zipCode"),
                    get_address_field(business, "street1"),
                    get_address_field(business, "street2"),
                    get_address_field(business, "city"),
                    get_address_field(business, "stateOrCountry"),
                    get_address_field(business, "stateOrCountryDescription"),
                    get_address_field(business, "zipCode"),
                ])
                entity_rows += 1

                # ------------------------------------------------
                # former names
                # ------------------------------------------------
                if isinstance(former_names, list):
                    for item in former_names:
                        if not isinstance(item, dict):
                            continue
                        former_writer.writerow([
                            rel_path,
                            cik,
                            "" if item.get("name") is None else str(item.get("name")),
                            "" if item.get("from") is None else str(item.get("from")),
                            "" if item.get("to") is None else str(item.get("to")),
                        ])
                        former_name_rows += 1

                # ------------------------------------------------
                # filings.files
                # ------------------------------------------------
                if isinstance(filing_files, list):
                    for item in filing_files:
                        if not isinstance(item, dict):
                            continue
                        file_writer.writerow([
                            rel_path,
                            cik,
                            "" if item.get("name") is None else str(item.get("name")),
                            "" if item.get("filingCount") is None else str(item.get("filingCount")),
                            "" if item.get("filingFrom") is None else str(item.get("filingFrom")),
                            "" if item.get("filingTo") is None else str(item.get("filingTo")),
                        ])
                        filing_file_rows += 1

                # ------------------------------------------------
                # filings.recent
                # ------------------------------------------------
                n_recent_filings = 0
                earliest_filing_date = ""
                latest_filing_date = ""

                if isinstance(recent, dict) and recent:
                    lengths = [len(v) for v in recent.values() if isinstance(v, list)]
                    n_recent_filings = max(lengths) if lengths else 0

                    accession_numbers = recent.get("accessionNumber", [])
                    filing_dates = recent.get("filingDate", [])
                    report_dates = recent.get("reportDate", [])
                    acceptance_times = recent.get("acceptanceDateTime", [])
                    acts = recent.get("act", [])
                    forms = recent.get("form", [])
                    file_numbers = recent.get("fileNumber", [])
                    film_numbers = recent.get("filmNumber", [])
                    items = recent.get("items", [])
                    sizes = recent.get("size", [])
                    is_xbrl = recent.get("isXBRL", [])
                    is_inline_xbrl = recent.get("isInlineXBRL", [])
                    primary_docs = recent.get("primaryDocument", [])
                    primary_doc_descs = recent.get("primaryDocDescription", [])

                    for i in range(n_recent_filings):
                        filing_date = safe_list_item(filing_dates, i)
                        earliest_filing_date = safe_min_date(earliest_filing_date, filing_date)
                        latest_filing_date = safe_max_date(latest_filing_date, filing_date)

                        recent_writer.writerow([
                            rel_path,
                            cik,
                            safe_list_item(accession_numbers, i),
                            filing_date,
                            safe_list_item(report_dates, i),
                            safe_list_item(acceptance_times, i),
                            safe_list_item(acts, i),
                            safe_list_item(forms, i),
                            safe_list_item(file_numbers, i),
                            safe_list_item(film_numbers, i),
                            safe_list_item(items, i),
                            safe_list_item(sizes, i),
                            safe_list_item(is_xbrl, i),
                            safe_list_item(is_inline_xbrl, i),
                            safe_list_item(primary_docs, i),
                            safe_list_item(primary_doc_descs, i),
                        ])
                        recent_rows += 1

                # ------------------------------------------------
                # inventory row
                # ------------------------------------------------
                inv_writer.writerow([
                    rel_path,
                    file_size,
                    cik,
                    entity_name,
                    entity_type,
                    n_tickers,
                    n_exchanges,
                    n_former_names,
                    n_recent_filings,
                    n_file_refs,
                    earliest_filing_date,
                    latest_filing_date,
                    "ok",
                    "",
                ])
                inventory_rows += 1
                ok_files += 1

            except Exception as exc:
                failed_files += 1
                error_rows += 1

                err_writer.writerow([
                    rel_path,
                    type(exc).__name__,
                    str(exc),
                ])

                inv_writer.writerow([
                    rel_path,
                    file_size,
                    "",
                    "",
                    "",
                    0,
                    0,
                    0,
                    0,
                    0,
                    "",
                    "",
                    "error",
                    str(exc),
                ])
                inventory_rows += 1

        inv_fh.flush()
        err_fh.flush()
        ent_fh.flush()
        recent_fh.flush()
        file_fh.flush()
        former_fh.flush()

    return {
        "shard_id": shard_id,
        "inventory_part": str(inventory_part_path),
        "errors_part": str(errors_part_path),
        "entities_part": str(entities_part_path),
        "recent_filings_part": str(recent_filings_part_path),
        "filing_files_part": str(filing_files_part_path),
        "former_names_part": str(former_names_part_path),
        "input_files": len(files),
        "ok_files": ok_files,
        "failed_files": failed_files,
        "inventory_rows": inventory_rows,
        "error_rows": error_rows,
        "entity_rows": entity_rows,
        "recent_rows": recent_rows,
        "filing_file_rows": filing_file_rows,
        "former_name_rows": former_name_rows,
    }


# ============================================================
# ARGUMENTS
# ============================================================

def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Inventory + flatten SEC submissions JSON files into partitioned CSVs."
    )
    parser.add_argument(
        "--input-dir",
        type=str,
        default=str(RAW_INPUT_DIR),
        help="Directory containing extracted SEC submissions JSON files.",
    )
    parser.add_argument(
        "--workers",
        type=int,
        default=0,
        help="Number of worker processes. 0 = auto.",
    )
    parser.add_argument(
        "--shard-multiplier",
        type=int,
        default=4,
        help="Approximate shards = workers * shard_multiplier.",
    )
    parser.add_argument(
        "--overwrite",
        action="store_true",
        help="Delete previous processed submissions outputs before running.",
    )
    parser.add_argument(
        "--resume",
        action="store_true",
        help="Resume mode: skip files already recorded in existing inventory or inventory shards.",
    )
    parser.add_argument(
        "--min-free-gb",
        type=float,
        default=8.0,
        help="Minimum free disk space to keep available. If free space drops below this, stop gracefully.",
    )
    parser.add_argument(
        "--disk-check-every-files",
        type=int,
        default=32,
        help="Worker disk space check frequency (every N files).",
    )
    return parser.parse_args()


# ============================================================
# MAIN
# ============================================================

def main() -> None:
    args = parse_args()

    input_dir = Path(args.input_dir)
    workers = choose_worker_count(args.workers)
    min_free_bytes = int(args.min_free_gb * (1024 ** 3))

    ensure_dirs(args.overwrite)

    print(f"Input directory : {input_dir}", flush=True)
    print(f"Output root     : {OUT_ROOT}", flush=True)
    print(f"JSON parser     : {'orjson' if HAS_ORJSON else 'stdlib json'}", flush=True)
    print(f"Workers         : {workers}", flush=True)
    print(f"Resume mode     : {args.resume}", flush=True)
    print(f"Min free space  : {args.min_free_gb:.2f} GB", flush=True)

    overall_start = now_ts()

    # --------------------------------------------------------
    # Discover files
    # --------------------------------------------------------
    discover_start = now_ts()
    all_files = discover_json_files(input_dir)
    all_input_bytes = sum(p.stat().st_size for p in all_files)
    discover_elapsed = now_ts() - discover_start

    print(
        f"Discovered {len(all_files):,} submissions JSON files "
        f"({human_bytes(all_input_bytes)}) in {fmt_elapsed(discover_elapsed)}",
        flush=True,
    )

    current_free = free_bytes(OUT_ROOT)
    print(f"Current free disk: {human_bytes(current_free)}", flush=True)
    if current_free < min_free_bytes:
        raise SystemExit(
            f"Refusing to start: free disk space {human_bytes(current_free)} is already below "
            f"the required minimum of {human_bytes(min_free_bytes)}."
        )

    # --------------------------------------------------------
    # Incremental resume filtering
    # --------------------------------------------------------
    previous_summary = load_previous_summary(SUMMARY_FINAL)

    processed_relpaths: set[str] = set()
    processed_basenames: set[str] = set()
    files = all_files
    skipped_existing = 0

    if args.resume:
        processed_relpaths, processed_basenames = load_existing_processed_keys()

        filtered: list[Path] = []
        for p in all_files:
            rel = str(p.relative_to(input_dir)).replace("\\", "/")
            base = p.name
            if rel in processed_relpaths or base in processed_basenames:
                skipped_existing += 1
                continue
            filtered.append(p)

        files = filtered

        print(
            f"Existing processed files detected: {len(processed_basenames):,} "
            f"(skipping {skipped_existing:,} already processed files)",
            flush=True,
        )

    if not files:
        print("No new submissions JSON files to process. Nothing to do.", flush=True)
        return

    new_input_bytes = sum(p.stat().st_size for p in files)
    print(
        f"New files to process: {len(files):,} "
        f"({human_bytes(new_input_bytes)})",
        flush=True,
    )

    # --------------------------------------------------------
    # Partition into shards
    # --------------------------------------------------------
    shard_count = max(1, min(len(files), workers * max(1, args.shard_multiplier)))
    shards = greedy_partition(files, shard_count)
    first_part_id = next_available_part_id(PARTS_DIR)

    print(f"Shard count       : {len(shards)}", flush=True)
    print(f"Starting part id  : {first_part_id:05d}", flush=True)
    print(f"Parts directory   : {PARTS_DIR}", flush=True)

    # --------------------------------------------------------
    # Process shards in parallel
    # --------------------------------------------------------
    futures = []
    shard_start = now_ts()

    total_files_done = 0
    total_ok_files = 0
    total_failed_files = 0
    total_inventory_rows = 0
    total_error_rows = 0
    total_entity_rows = 0
    total_recent_rows = 0
    total_filing_file_rows = 0
    total_former_name_rows = 0

    executor = ProcessPoolExecutor(max_workers=workers)
    try:
        for offset, shard_files in enumerate(shards):
            shard_id = first_part_id + offset
            futures.append(
                executor.submit(
                    process_shard,
                    shard_id,
                    [str(p) for p in shard_files],
                    str(input_dir),
                    str(PARTS_DIR),
                    min_free_bytes,
                    args.disk_check_every_files,
                )
            )

        completed = 0
        total_shards = len(futures)

        for fut in as_completed(futures):
            result = fut.result()
            completed += 1

            total_files_done += result["input_files"]
            total_ok_files += result["ok_files"]
            total_failed_files += result["failed_files"]
            total_inventory_rows += result["inventory_rows"]
            total_error_rows += result["error_rows"]
            total_entity_rows += result["entity_rows"]
            total_recent_rows += result["recent_rows"]
            total_filing_file_rows += result["filing_file_rows"]
            total_former_name_rows += result["former_name_rows"]

            elapsed = max(now_ts() - shard_start, 1e-9)
            files_per_sec = total_files_done / elapsed
            recent_rows_per_sec = total_recent_rows / elapsed if total_recent_rows else 0.0

            print(
                f"[process] shards {completed}/{total_shards} | "
                f"new files {total_files_done:,}/{len(files):,} | "
                f"ok {total_ok_files:,} | failed {total_failed_files:,} | "
                f"recent rows {total_recent_rows:,} | "
                f"{files_per_sec:,.1f} files/s | {recent_rows_per_sec:,.1f} rows/s",
                flush=True,
            )

    except KeyboardInterrupt:
        print(
            "\nInterrupted. Partial shard outputs are preserved. "
            "Re-run with --resume to continue from completed work.",
            flush=True,
        )
        executor.shutdown(wait=False, cancel_futures=True)
        raise

    except LowDiskSpaceError as exc:
        print(
            f"\nStopped due to low disk space: {exc}\n"
            f"Partial shard outputs are preserved. Make space and re-run with --resume.",
            flush=True,
        )
        executor.shutdown(wait=False, cancel_futures=True)
        return

    except Exception:
        executor.shutdown(wait=False, cancel_futures=True)
        raise

    finally:
        try:
            executor.shutdown(wait=True, cancel_futures=False)
        except Exception:
            pass

    process_elapsed = now_ts() - shard_start

    # --------------------------------------------------------
    # Merge inventory and errors only
    # --------------------------------------------------------
    inventory_parts = sorted(PARTS_DIR.glob("inventory_part_*.csv"))
    error_parts = sorted(PARTS_DIR.glob("errors_part_*.csv"))

    print("Rebuilding cumulative inventory CSV...", flush=True)
    merge_csv_parts(inventory_parts, INVENTORY_FINAL, "[merge inventory]")

    print("Rebuilding cumulative errors CSV...", flush=True)
    merge_csv_parts(error_parts, ERRORS_FINAL, "[merge errors]")

    # --------------------------------------------------------
    # Other outputs stay partitioned
    # --------------------------------------------------------
    entities_parts = sorted(PARTS_DIR.glob("entities_part_*.csv"))
    recent_parts = sorted(PARTS_DIR.glob("recent_filings_part_*.csv"))
    file_ref_parts = sorted(PARTS_DIR.glob("filing_files_part_*.csv"))
    former_name_parts = sorted(PARTS_DIR.glob("former_names_part_*.csv"))

    previous_total_recent_rows = int(previous_summary.get("total_recent_rows", 0) or 0)
    previous_total_entity_rows = int(previous_summary.get("total_entity_rows", 0) or 0)
    previous_total_filing_file_rows = int(previous_summary.get("total_filing_file_rows", 0) or 0)
    previous_total_former_name_rows = int(previous_summary.get("total_former_name_rows", 0) or 0)

    summary = {
        "input_dir": str(input_dir),
        "output_root": str(OUT_ROOT),
        "used_orjson": HAS_ORJSON,
        "workers": workers,
        "resume_mode": args.resume,
        "min_free_gb": args.min_free_gb,
        "shard_count_this_run": len(shards),
        "starting_part_id_this_run": first_part_id,
        "json_files_discovered_total": len(all_files),
        "json_files_processed_this_run": len(files),
        "json_files_skipped_as_existing": skipped_existing,
        "total_input_bytes_all_visible": all_input_bytes,
        "total_input_bytes_processed_this_run": new_input_bytes,
        "total_ok_files_this_run": total_ok_files,
        "total_failed_files_this_run": total_failed_files,
        "total_inventory_rows_this_run": total_inventory_rows,
        "total_error_rows_this_run": total_error_rows,
        "total_entity_rows_this_run": total_entity_rows,
        "total_recent_rows_this_run": total_recent_rows,
        "total_filing_file_rows_this_run": total_filing_file_rows,
        "total_former_name_rows_this_run": total_former_name_rows,
        "total_entity_rows": previous_total_entity_rows + total_entity_rows,
        "total_recent_rows": previous_total_recent_rows + total_recent_rows,
        "total_filing_file_rows": previous_total_filing_file_rows + total_filing_file_rows,
        "total_former_name_rows": previous_total_former_name_rows + total_former_name_rows,
        "entities_parts": [str(p) for p in entities_parts],
        "recent_filings_parts": [str(p) for p in recent_parts],
        "filing_files_parts": [str(p) for p in file_ref_parts],
        "former_names_parts": [str(p) for p in former_name_parts],
        "inventory_csv": str(INVENTORY_FINAL),
        "errors_csv": str(ERRORS_FINAL),
        "timing": {
            "processing_seconds_this_run": process_elapsed,
            "total_seconds_this_run": now_ts() - overall_start,
        },
    }

    with SUMMARY_FINAL.open("w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    total_elapsed = now_ts() - overall_start

    print("\n=== SUBMISSIONS PIPELINE COMPLETE ===", flush=True)
    print(f"Inventory CSV    : {INVENTORY_FINAL}", flush=True)
    print(f"Errors CSV       : {ERRORS_FINAL}", flush=True)
    print(f"Entity parts     : {len(entities_parts)} file(s)", flush=True)
    print(f"Recent parts     : {len(recent_parts)} file(s)", flush=True)
    print(f"File-ref parts   : {len(file_ref_parts)} file(s)", flush=True)
    print(f"Former-name parts: {len(former_name_parts)} file(s)", flush=True)
    print(f"Summary JSON     : {SUMMARY_FINAL}", flush=True)
    print(f"All visible JSON : {len(all_files):,}", flush=True)
    print(f"Processed this run: {len(files):,}", flush=True)
    print(f"Skipped existing : {skipped_existing:,}", flush=True)
    print(f"Recent filing rows this run : {total_recent_rows:,}", flush=True)
    print(f"Cumulative recent rows      : {previous_total_recent_rows + total_recent_rows:,}", flush=True)
    print(f"Elapsed           : {fmt_elapsed(total_elapsed)}", flush=True)


if __name__ == "__main__":
    main()

# python data/sec_submissions_pipeline.py --workers 8 --min-free-gb 8
# Resume after interruption / low disk:
# python data/sec_submissions_pipeline.py --workers 8 --resume --min-free-gb 8

### data\sec_issuer_master.py

In [ ]:
#!/usr/bin/env python3
"""
SEC Issuer Master Builder

Builds a clean issuer/company master table from processed SEC submissions and companyfacts
inventories. Filters for U.S. issuers and produces a canonical CIK-to-ticker mapping
for downstream fundamental and text processing.

Inputs:
- submissions_inventory.csv (from sec_submissions_pipeline.py)
- companyfacts_inventory.csv (from sec_companyfacts_pipeline.py)
- Optional: entities_part_*.csv for richer metadata

Outputs:
- issuer_master.csv: canonical issuer table with U.S. flag
- cik_ticker_map.csv: CIK to primary ticker mapping
- issuer_master_summary.json
"""

from __future__ import annotations

import argparse
import csv
import json
import os
import re
import time
from pathlib import Path
from typing import Any, Optional

# ============================================================
# PATHS
# ============================================================

dataPath = Path(os.getenv("dataPathGlobal", "data"))

SUBMISSIONS_ROOT = dataPath / "sec_edgar" / "processed" / "submissions"
COMPANYFACTS_ROOT = dataPath / "sec_edgar" / "processed" / "companyfacts"
OUT_ROOT = dataPath / "sec_edgar" / "processed" / "issuer_master"

SUBMISSIONS_INVENTORY = SUBMISSIONS_ROOT / "submissions_inventory.csv"
COMPANYFACTS_INVENTORY = COMPANYFACTS_ROOT / "companyfacts_inventory.csv"
ENTITIES_PARTS_DIR = SUBMISSIONS_ROOT / "submissions_flat"

ISSUER_MASTER_FINAL = OUT_ROOT / "issuer_master.csv"
CIK_TICKER_MAP_FINAL = OUT_ROOT / "cik_ticker_map.csv"
SUMMARY_FINAL = OUT_ROOT / "issuer_master_summary.json"

# U.S. state/territory codes for filtering
US_STATES = {
    "AL", "AK", "AZ", "AR", "CA", "CO", "CT", "DE", "FL", "GA",
    "HI", "ID", "IL", "IN", "IA", "KS", "KY", "LA", "ME", "MD",
    "MA", "MI", "MN", "MS", "MO", "MT", "NE", "NV", "NH", "NJ",
    "NM", "NY", "NC", "ND", "OH", "OK", "OR", "PA", "RI", "SC",
    "SD", "TN", "TX", "UT", "VT", "VA", "WA", "WV", "WI", "WY",
    "DC", "PR", "VI", "GU", "AS", "MP",
}

# Common non-U.S. incorporation keywords to filter out
NON_US_KEYWORDS = {
    "CANADA", "CAYMAN", "BERMUDA", "BRITISH", "VIRGIN ISLANDS",
    "IRELAND", "UNITED KINGDOM", "UK", "ENGLAND", "WALES", "SCOTLAND",
    "NETHERLANDS", "LUXEMBOURG", "SWITZERLAND", "FRANCE", "GERMANY",
    "ISRAEL", "CHINA", "HONG KONG", "SINGAPORE", "JAPAN", "AUSTRALIA",
    "NEW ZEALAND", "INDIA", "BRAZIL", "MEXICO", "PANAMA", "BAHAMAS",
    "BARBADOS", "CURACAO", "JERSEY", "GUERNSEY", "ISLE OF MAN",
    "MARSHALL ISLANDS", "MAURITIUS", "SEYCHELLES", "LIBERIA",
}


# ============================================================
# HELPERS
# ============================================================

def now_ts() -> float:
    return time.time()


def fmt_elapsed(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.1f}s"
    minutes = seconds / 60.0
    if minutes < 60:
        return f"{minutes:.1f}m"
    hours = minutes / 60.0
    return f"{hours:.2f}h"


def ensure_dirs() -> None:
    OUT_ROOT.mkdir(parents=True, exist_ok=True)


def parse_pipe_list(value: str) -> list[str]:
    """Parse pipe-separated list from submissions output."""
    if not value:
        return []
    return [v.strip() for v in value.split("|") if v.strip()]


def is_us_issuer(
    state_of_incorporation: str,
    state_desc: str,
    mailing_state: str,
    business_state: str,
) -> bool:
    """
    Determine if issuer is U.S.-based using incorporation and address signals.
    Returns True if likely U.S. issuer, False otherwise.
    """
    # Check incorporation state
    if state_of_incorporation and state_of_incorporation.upper() in US_STATES:
        return True

    # Check incorporation description for non-US keywords
    if state_desc:
        state_desc_upper = state_desc.upper()
        if any(kw in state_desc_upper for kw in NON_US_KEYWORDS):
            return False

    # Check mailing/business address states
    if mailing_state and mailing_state.upper() in US_STATES:
        return True
    if business_state and business_state.upper() in US_STATES:
        return True

    # If we have a state description with US state, it's US
    if state_desc:
        for state in US_STATES:
            if f", {state}" in state_desc or f" {state} " in state_desc:
                return True

    return False


def load_submissions_inventory() -> dict[str, dict[str, Any]]:
    """
    Load submissions inventory and index by CIK.
    Returns: dict[cik] -> inventory row dict
    """
    if not SUBMISSIONS_INVENTORY.exists():
        print(f"Warning: {SUBMISSIONS_INVENTORY} not found. Proceeding without submissions metadata.")
        return {}

    cik_map = {}
    with SUBMISSIONS_INVENTORY.open("r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            cik = row.get("cik", "").strip()
            status = row.get("status", "")
            if cik and status == "ok":
                # Keep the first (or merge logic can go here)
                if cik not in cik_map:
                    cik_map[cik] = row
    return cik_map


def load_companyfacts_inventory() -> dict[str, dict[str, Any]]:
    """
    Load companyfacts inventory and index by CIK.
    Returns: dict[cik] -> inventory row dict
    """
    if not COMPANYFACTS_INVENTORY.exists():
        print(f"Warning: {COMPANYFACTS_INVENTORY} not found. Proceeding without companyfacts metadata.")
        return {}

    cik_map = {}
    with COMPANYFACTS_INVENTORY.open("r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            cik = row.get("cik", "").strip()
            status = row.get("status", "")
            if cik and status == "ok":
                if cik not in cik_map:
                    cik_map[cik] = row
    return cik_map


def load_entities_metadata() -> dict[str, dict[str, Any]]:
    """
    Load entities metadata from partitioned entities parts.
    Returns: dict[cik] -> entity row dict (latest or merged)
    """
    entities_parts = sorted(ENTITIES_PARTS_DIR.glob("entities_part_*.csv"))
    if not entities_parts:
        print(f"Warning: No entities parts found in {ENTITIES_PARTS_DIR}")
        return {}

    cik_map = {}
    for part_path in entities_parts:
        with part_path.open("r", newline="", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                cik = row.get("cik", "").strip()
                if not cik:
                    continue

                # Keep the most recent by source_json timestamp or first seen
                if cik not in cik_map:
                    cik_map[cik] = row
                else:
                    # Prefer the one with more complete data
                    existing = cik_map[cik]
                    if len(row.get("entityType", "")) > len(existing.get("entityType", "")):
                        cik_map[cik] = row

    return cik_map


def extract_primary_ticker(tickers_str: str, exchanges_str: str) -> tuple[str, str]:
    """
    Extract primary ticker from pipe-separated strings.
    Returns: (primary_ticker, primary_exchange)
    Prioritizes NYSE/NASDAQ over OTC/OTHER.
    """
    tickers = parse_pipe_list(tickers_str)
    exchanges = parse_pipe_list(exchanges_str)

    if not tickers:
        return "", ""

    # Preferred exchange order (rough heuristic)
    exchange_priority = {
        "NYSE": 1, "New York Stock Exchange": 1,
        "NASDAQ": 2, "Nasdaq": 2,
        "AMEX": 3, "NYSE American": 3,
        "OTC": 10, "OTCBB": 10, "Other OTC": 10,
    }

    best_idx = 0
    best_priority = 999

    for i, (ticker, exchange) in enumerate(zip(tickers, exchanges)):
        if i >= len(tickers):
            break
        priority = exchange_priority.get(exchange, 50)
        if priority < best_priority:
            best_priority = priority
            best_idx = i

    primary_ticker = tickers[best_idx] if best_idx < len(tickers) else tickers[0]
    primary_exchange = exchanges[best_idx] if best_idx < len(exchanges) else ""

    return primary_ticker, primary_exchange


def build_issuer_master() -> tuple[list[dict], dict]:
    """
    Build consolidated issuer master table.
    Returns: (issuer_rows, stats_dict)
    """
    print("Loading submissions inventory...", flush=True)
    submissions_map = load_submissions_inventory()

    print("Loading companyfacts inventory...", flush=True)
    companyfacts_map = load_companyfacts_inventory()

    print("Loading entities metadata...", flush=True)
    entities_map = load_entities_metadata()

    # Combine all unique CIKs
    all_ciks = set(submissions_map.keys()) | set(companyfacts_map.keys()) | set(entities_map.keys())
    print(f"Total unique CIKs across sources: {len(all_ciks):,}", flush=True)

    issuer_rows = []
    us_count = 0
    non_us_count = 0
    unknown_count = 0
    with_ticker = 0

    # Process in batches for progress
    ciks_sorted = sorted(all_ciks)
    total_ciks = len(ciks_sorted)
    progress_interval = max(1, total_ciks // 20)

    for idx, cik in enumerate(ciks_sorted):
        if idx % progress_interval == 0:
            pct = (idx / total_ciks) * 100
            print(f"  Processing CIKs: {idx:,}/{total_ciks:,} ({pct:.1f}%)", flush=True)

        sub_row = submissions_map.get(cik, {})
        fact_row = companyfacts_map.get(cik, {})
        entity_row = entities_map.get(cik, {})

        # Determine entity name
        entity_name = (
            entity_row.get("name") or
            sub_row.get("entity_name") or
            fact_row.get("entity_name") or
            ""
        )

        # U.S. issuer determination
        state_incorp = entity_row.get("stateOfIncorporation", "")
        state_desc = entity_row.get("stateOfIncorporationDescription", "")
        mailing_state = entity_row.get("mailing_stateOrCountry", "")
        business_state = entity_row.get("business_stateOrCountry", "")

        is_us = is_us_issuer(state_incorp, state_desc, mailing_state, business_state)

        if is_us:
            us_count += 1
        elif state_incorp or state_desc:
            non_us_count += 1
        else:
            unknown_count += 1

        # Extract primary ticker
        tickers_str = entity_row.get("tickers", "")
        exchanges_str = entity_row.get("exchanges", "")
        primary_ticker, primary_exchange = extract_primary_ticker(tickers_str, exchanges_str)

        if primary_ticker:
            with_ticker += 1

        # Build row
        row = {
            "cik": cik,
            "padded_cik": cik.zfill(10),
            "entity_name": entity_name,
            "entity_type": entity_row.get("entityType", ""),
            "is_us_issuer": "1" if is_us else "0",
            "primary_ticker": primary_ticker,
            "primary_exchange": primary_exchange,
            "all_tickers": tickers_str,
            "sic": entity_row.get("sic", ""),
            "sic_description": entity_row.get("sicDescription", ""),
            "fiscal_year_end": entity_row.get("fiscalYearEnd", ""),
            "state_of_incorporation": state_incorp,
            "state_of_incorporation_desc": state_desc,
            "business_city": entity_row.get("business_city", ""),
            "business_state": business_state,
            "mailing_city": entity_row.get("mailing_city", ""),
            "mailing_state": mailing_state,
            "has_submissions": "1" if cik in submissions_map else "0",
            "has_companyfacts": "1" if cik in companyfacts_map else "0",
            "earliest_filing_sub": sub_row.get("earliest_filing_date", ""),
            "latest_filing_sub": sub_row.get("latest_filing_date", ""),
            "earliest_filed_facts": fact_row.get("earliest_filed", ""),
            "latest_filed_facts": fact_row.get("latest_filed", ""),
            "n_fact_observations": fact_row.get("n_observations", "0"),
        }
        issuer_rows.append(row)

    stats = {
        "total_ciks": total_ciks,
        "us_issuers": us_count,
        "non_us_issuers": non_us_count,
        "unknown_jurisdiction": unknown_count,
        "issuers_with_ticker": with_ticker,
    }

    return issuer_rows, stats


def write_issuer_master(issuer_rows: list[dict]) -> None:
    """Write issuer master CSV."""
    if not issuer_rows:
        print("No issuer rows to write.")
        return

    fieldnames = list(issuer_rows[0].keys())

    tmp_path = ISSUER_MASTER_FINAL.with_suffix(".csv.part")
    with tmp_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(issuer_rows)

    tmp_path.replace(ISSUER_MASTER_FINAL)
    print(f"Written: {ISSUER_MASTER_FINAL}", flush=True)


def write_cik_ticker_map(issuer_rows: list[dict]) -> None:
    """Write CIK to primary ticker mapping for U.S. issuers only."""
    cik_ticker_rows = []

    for row in issuer_rows:
        if row["is_us_issuer"] == "1" and row["primary_ticker"]:
            cik_ticker_rows.append({
                "cik": row["cik"],
                "padded_cik": row["padded_cik"],
                "primary_ticker": row["primary_ticker"],
                "entity_name": row["entity_name"],
                "primary_exchange": row["primary_exchange"],
            })

    if not cik_ticker_rows:
        print("No U.S. issuers with tickers to write.")
        return

    fieldnames = ["cik", "padded_cik", "primary_ticker", "entity_name", "primary_exchange"]

    tmp_path = CIK_TICKER_MAP_FINAL.with_suffix(".csv.part")
    with tmp_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(cik_ticker_rows)

    tmp_path.replace(CIK_TICKER_MAP_FINAL)
    print(f"Written: {CIK_TICKER_MAP_FINAL}", flush=True)
    print(f"  U.S. issuers with tickers: {len(cik_ticker_rows):,}", flush=True)


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Build SEC issuer master table with U.S. filtering."
    )
    parser.add_argument(
        "--overwrite",
        action="store_true",
        help="Overwrite existing outputs.",
    )
    args = parser.parse_args()

    ensure_dirs()

    # Check if outputs already exist (resume behavior)
    if not args.overwrite and ISSUER_MASTER_FINAL.exists() and CIK_TICKER_MAP_FINAL.exists():
        print(f"Outputs already exist. Use --overwrite to regenerate.")
        print(f"  {ISSUER_MASTER_FINAL}")
        print(f"  {CIK_TICKER_MAP_FINAL}")
        return

    overall_start = now_ts()

    print("=" * 60, flush=True)
    print("SEC ISSUER MASTER BUILDER", flush=True)
    print("=" * 60, flush=True)

    # Build issuer master
    build_start = now_ts()
    issuer_rows, stats = build_issuer_master()
    build_elapsed = now_ts() - build_start

    print(f"\nBuild completed in {fmt_elapsed(build_elapsed)}", flush=True)
    print(f"\nStatistics:", flush=True)
    print(f"  Total unique CIKs: {stats['total_ciks']:,}", flush=True)
    print(f"  U.S. issuers: {stats['us_issuers']:,}", flush=True)
    print(f"  Non-U.S. issuers: {stats['non_us_issuers']:,}", flush=True)
    print(f"  Unknown jurisdiction: {stats['unknown_jurisdiction']:,}", flush=True)
    print(f"  Issuers with ticker: {stats['issuers_with_ticker']:,}", flush=True)

    # Write outputs
    print("\nWriting outputs...", flush=True)
    write_issuer_master(issuer_rows)
    write_cik_ticker_map(issuer_rows)

    # Write summary
    summary = {
        "output_root": str(OUT_ROOT),
        "issuer_master": str(ISSUER_MASTER_FINAL),
        "cik_ticker_map": str(CIK_TICKER_MAP_FINAL),
        "statistics": stats,
        "timing": {
            "build_seconds": build_elapsed,
            "total_seconds": now_ts() - overall_start,
        },
    }

    with SUMMARY_FINAL.open("w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    total_elapsed = now_ts() - overall_start
    print(f"\n=== ISSUER MASTER COMPLETE ===", flush=True)
    print(f"Summary JSON: {SUMMARY_FINAL}", flush=True)
    print(f"Total elapsed: {fmt_elapsed(total_elapsed)}", flush=True)


if __name__ == "__main__":
    main()

# pyhton data/sec_issuer_master.py

### data\sec_core_fundamentals.py

In [ ]:
#!/usr/bin/env python3
"""
SEC Core Fundamentals Extractor (Strict Point-in-Time)

Extracts a clean, point-in-time quarterly fundamentals table from the flattened
SEC companyfacts table. Uses strict "as-first-reported" logic to prevent lookahead bias.

Inputs:
- companyfacts_flat/facts_part_*.csv (121M+ rows)
- issuer_master/cik_ticker_map.csv (for U.S. issuer filtering)

Outputs:
- core_fundamentals_quarterly.csv: Clean quarterly fundamentals table
- fundamentals_summary.json
"""

from __future__ import annotations

import argparse
import csv
import heapq
import json
import os
import re
import time
from collections import defaultdict
from pathlib import Path
from typing import Any, Optional

# ============================================================
# PATHS
# ============================================================

dataPath = Path(os.getenv("dataPathGlobal", "data"))

FACTS_PARTS_DIR = dataPath / "sec_edgar" / "processed" / "companyfacts" / "companyfacts_flat"
ISSUER_MASTER_DIR = dataPath / "sec_edgar" / "processed" / "issuer_master"
OUT_ROOT = dataPath / "sec_edgar" / "processed" / "fundamentals"

CIK_TICKER_MAP = ISSUER_MASTER_DIR / "cik_ticker_map.csv"
CORE_FUNDAMENTALS_FINAL = OUT_ROOT / "core_fundamentals_quarterly.csv"
SUMMARY_FINAL = OUT_ROOT / "fundamentals_summary.json"

TMP_DIR = OUT_ROOT / "_tmp"
PARTS_DIR = OUT_ROOT / "fundamentals_parts"

# ============================================================
# CONCEPT MAPPING (SEC XBRL tags to standard names)
# ============================================================

# Format: (priority_list_of_xbrl_tags, output_column_name, aggregation_mode)
# aggregation_mode: "sum", "first", or "latest_filed" (for point-in-time)
CONCEPT_MAP = [
    # Income Statement
    (["Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax",
      "SalesRevenueNet", "SalesRevenueGoodsNet", "RevenueFromContractWithCustomer"],
     "revenue", "sum"),
    (["CostOfRevenue", "CostOfGoodsAndServicesSold", "CostOfSales"],
     "cost_of_revenue", "sum"),
    (["GrossProfit"], "gross_profit", "first"),
    (["OperatingExpenses", "OperatingExpense"], "operating_expenses", "sum"),
    (["OperatingIncomeLoss", "OperatingIncome"], "operating_income", "first"),
    (["NetIncomeLoss", "NetIncome", "ProfitLoss"],
     "net_income", "first"),
    (["EarningsPerShareBasic", "EarningsPerShare"],
     "eps_basic", "first"),
    (["EarningsPerShareDiluted"], "eps_diluted", "first"),
    (["WeightedAverageNumberOfSharesOutstandingBasic",
      "WeightedAverageNumberOfDilutedSharesOutstanding"],
     "shares_basic", "first"),

    # Balance Sheet - Assets
    (["Assets", "TotalAssets"], "total_assets", "first"),
    (["AssetsCurrent", "CurrentAssets"], "current_assets", "first"),
    (["CashAndCashEquivalentsAtCarryingValue", "CashAndCashEquivalents",
      "Cash"], "cash_and_equivalents", "first"),
    (["InventoryNet", "Inventory"], "inventory", "first"),
    (["PropertyPlantAndEquipmentNet", "PropertyPlantAndEquipment"],
     "ppe_net", "first"),
    (["Goodwill"], "goodwill", "first"),
    (["IntangibleAssetsNetExcludingGoodwill", "IntangibleAssetsNet"],
     "intangible_assets", "first"),

    # Balance Sheet - Liabilities
    (["Liabilities", "TotalLiabilities"], "total_liabilities", "first"),
    (["LiabilitiesCurrent", "CurrentLiabilities"], "current_liabilities", "first"),
    (["LongTermDebt", "LongTermDebtNoncurrent", "LongTermDebtAndCapitalLeaseObligations"],
     "long_term_debt", "first"),
    (["DebtCurrent", "ShortTermBorrowings"], "short_term_debt", "first"),

    # Balance Sheet - Equity
    (["StockholdersEquity", "ShareholdersEquity", "Equity"],
     "shareholders_equity", "first"),
    (["RetainedEarningsAccumulatedDeficit", "RetainedEarnings"],
     "retained_earnings", "first"),

    # Cash Flow
    (["NetCashProvidedByUsedInOperatingActivities", "OperatingCashFlow"],
     "operating_cash_flow", "first"),
    (["NetCashProvidedByUsedInInvestingActivities"],
     "investing_cash_flow", "first"),
    (["NetCashProvidedByUsedInFinancingActivities"],
     "financing_cash_flow", "first"),
    (["PaymentsToAcquirePropertyPlantAndEquipment", "CapitalExpenditure"],
     "capex", "sum"),
    (["FreeCashFlow"], "free_cash_flow", "first"),
]

# Forms considered for point-in-time quarterly extraction
QUARTERLY_FORMS = {"10-Q", "10-Q/A", "10-K", "10-K/A"}


# ============================================================
# HELPERS
# ============================================================

def now_ts() -> float:
    return time.time()


def fmt_elapsed(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.1f}s"
    minutes = seconds / 60.0
    if minutes < 60:
        return f"{minutes:.1f}m"
    hours = minutes / 60.0
    return f"{hours:.2f}h"


def human_bytes(n: int | float) -> str:
    n = float(n)
    units = ["B", "KB", "MB", "GB", "TB"]
    for unit in units:
        if n < 1024.0 or unit == units[-1]:
            return f"{n:.2f} {unit}"
        n /= 1024.0
    return f"{n:.2f} B"


def ensure_dirs(overwrite: bool) -> None:
    OUT_ROOT.mkdir(parents=True, exist_ok=True)

    if overwrite:
        import shutil
        for p in [PARTS_DIR, TMP_DIR, CORE_FUNDAMENTALS_FINAL, SUMMARY_FINAL]:
            if p.is_dir():
                shutil.rmtree(p, ignore_errors=True)
            elif p.exists():
                p.unlink()

    PARTS_DIR.mkdir(parents=True, exist_ok=True)
    TMP_DIR.mkdir(parents=True, exist_ok=True)


def load_us_ciks() -> set[str]:
    """Load U.S. CIKs from cik_ticker_map."""
    if not CIK_TICKER_MAP.exists():
        print(f"Warning: {CIK_TICKER_MAP} not found. Processing all CIKs.")
        return set()

    us_ciks = set()
    with CIK_TICKER_MAP.open("r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            cik = row.get("cik", "").strip()
            if cik:
                us_ciks.add(cik)
    return us_ciks


def build_concept_lookup() -> dict[str, tuple[str, str]]:
    """
    Build mapping from SEC concept name to (output_column, aggregation_mode).
    """
    lookup = {}
    for tags, out_col, agg_mode in CONCEPT_MAP:
        for tag in tags:
            lookup[tag.lower()] = (out_col, agg_mode)
    return lookup


def parse_fiscal_period(fp: str) -> str:
    """
    Normalize fiscal period.
    SEC uses: FY, Q1, Q2, Q3, Q4, H1, H2, etc.
    Returns: "FY", "Q1", "Q2", "Q3", "Q4", or fp as-is.
    """
    fp_upper = fp.upper().strip()
    if fp_upper in ("FY", "Q1", "Q2", "Q3", "Q4"):
        return fp_upper
    return fp_upper


def fiscal_period_sort_key(fp: str) -> int:
    """Sort key for fiscal periods (FY comes after Q4)."""
    order = {"Q1": 1, "Q2": 2, "Q3": 3, "Q4": 4, "FY": 5}
    return order.get(fp, 99)


def process_facts_parts(us_ciks: set[str]) -> tuple[Path, dict]:
    """
    Process all facts parts, filter for U.S. CIKs, and extract point-in-time
    quarterly fundamentals.

    Returns: (output_path, stats_dict)
    """
    concept_lookup = build_concept_lookup()

    facts_parts = sorted(FACTS_PARTS_DIR.glob("facts_part_*.csv"))
    if not facts_parts:
        raise FileNotFoundError(f"No facts parts found in {FACTS_PARTS_DIR}")

    print(f"Found {len(facts_parts)} facts parts to process", flush=True)

    # We'll write to a single output file (could partition if needed)
    output_path = CORE_FUNDAMENTALS_FINAL

    fieldnames = [
        "cik",
        "ticker",
        "entity_name",
        "fiscal_year",
        "fiscal_period",
        "filing_date",
        "period_end_date",
        "form_type",
        "accession",
        "revenue",
        "cost_of_revenue",
        "gross_profit",
        "operating_expenses",
        "operating_income",
        "net_income",
        "eps_basic",
        "eps_diluted",
        "shares_basic",
        "total_assets",
        "current_assets",
        "cash_and_equivalents",
        "inventory",
        "ppe_net",
        "goodwill",
        "intangible_assets",
        "total_liabilities",
        "current_liabilities",
        "long_term_debt",
        "short_term_debt",
        "shareholders_equity",
        "retained_earnings",
        "operating_cash_flow",
        "investing_cash_flow",
        "financing_cash_flow",
        "capex",
        "free_cash_flow",
    ]

    tmp_path = output_path.with_suffix(".csv.part")
    total_rows = 0
    total_observations_processed = 0

    # Track which (cik, fy, fp) we've already written
    # For point-in-time, we want earliest filing date per period
    processed_periods: dict[tuple[str, str, str], dict] = {}

    with tmp_path.open("w", newline="", encoding="utf-8") as out_fh:
        writer = csv.DictWriter(out_fh, fieldnames=fieldnames)
        writer.writeheader()

        for part_idx, part_path in enumerate(facts_parts):
            part_size = part_path.stat().st_size
            print(f"\nProcessing part {part_idx+1}/{len(facts_parts)}: {part_path.name} ({human_bytes(part_size)})", flush=True)

            rows_in_part = 0
            observations_in_part = 0

            # Accumulate facts per (cik, fy, fp, concept)
            # Structure: {(cik, fy, fp): {concept: [(val, filed_date, form, end_date, accn)]}}
            period_facts: dict[tuple, dict] = defaultdict(lambda: defaultdict(list))

            with part_path.open("r", newline="", encoding="utf-8") as in_fh:
                reader = csv.DictReader(in_fh)

                for row in reader:
                    observations_in_part += 1

                    cik = row.get("cik", "").strip()
                    if not cik:
                        continue

                    # Filter for U.S. CIKs if we have the list
                    if us_ciks and cik not in us_ciks:
                        continue

                    form = (row.get("form") or "").strip().upper()
                    if form not in QUARTERLY_FORMS:
                        continue

                    fy = (row.get("fy") or "").strip()
                    fp = parse_fiscal_period(row.get("fp") or "")
                    if not fy or not fp:
                        continue

                    # Only quarterly/annual periods
                    if fp not in ("Q1", "Q2", "Q3", "Q4", "FY"):
                        continue

                    concept = (row.get("concept") or "").lower().strip()
                    if concept not in concept_lookup:
                        continue

                    val_str = (row.get("val") or "").strip()
                    if not val_str:
                        continue

                    try:
                        val = float(val_str)
                    except ValueError:
                        continue

                    filed = (row.get("filed") or "").strip()
                    end = (row.get("end") or "").strip()
                    accn = (row.get("accn") or "").strip()

                    key = (cik, fy, fp)
                    period_facts[key][concept].append((val, filed, form, end, accn))
                    rows_in_part += 1

            # Now for each period, select the earliest filing and aggregate facts
            for (cik, fy, fp), concept_dict in period_facts.items():
                # Determine the earliest filing across all concepts for this period
                earliest_filed = "9999-99-99"
                earliest_form = ""
                earliest_end = ""
                earliest_accn = ""

                for concept, obs_list in concept_dict.items():
                    for val, filed, form, end, accn in obs_list:
                        if filed < earliest_filed:
                            earliest_filed = filed
                            earliest_form = form
                            earliest_end = end
                            earliest_accn = accn

                # Now extract only facts from the earliest filing
                row_data = {
                    "cik": cik,
                    "ticker": "",  # Will join later if needed
                    "entity_name": "",
                    "fiscal_year": fy,
                    "fiscal_period": fp,
                    "filing_date": earliest_filed,
                    "period_end_date": earliest_end,
                    "form_type": earliest_form,
                    "accession": earliest_accn,
                }

                for concept, obs_list in concept_dict.items():
                    out_col, agg_mode = concept_lookup[concept]

                    # Filter to only observations from the earliest filing
                    earliest_obs = [v for v, f, _, _, _ in obs_list if f == earliest_filed]

                    if not earliest_obs:
                        continue

                    if agg_mode == "first":
                        row_data[out_col] = str(earliest_obs[0])
                    elif agg_mode == "sum":
                        row_data[out_col] = str(sum(earliest_obs))
                    else:
                        row_data[out_col] = str(earliest_obs[0])

                writer.writerow(row_data)
                total_rows += 1

            total_observations_processed += observations_in_part
            print(f"  Part rows written: {rows_in_part:,} | Cumulative output rows: {total_rows:,}", flush=True)

    tmp_path.replace(output_path)

    stats = {
        "total_output_rows": total_rows,
        "total_observations_processed": total_observations_processed,
        "us_ciks_filtered": len(us_ciks) if us_ciks else 0,
    }

    return output_path, stats


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Extract point-in-time quarterly fundamentals from SEC companyfacts."
    )
    parser.add_argument(
        "--overwrite",
        action="store_true",
        help="Overwrite existing outputs.",
    )
    args = parser.parse_args()

    ensure_dirs(args.overwrite)

    if not args.overwrite and CORE_FUNDAMENTALS_FINAL.exists():
        print(f"Output already exists: {CORE_FUNDAMENTALS_FINAL}")
        print("Use --overwrite to regenerate.")
        return

    overall_start = now_ts()

    print("=" * 60, flush=True)
    print("SEC CORE FUNDAMENTALS EXTRACTOR (Point-in-Time)", flush=True)
    print("=" * 60, flush=True)

    # Load U.S. CIKs for filtering
    print("\nLoading U.S. CIK list...", flush=True)
    us_ciks = load_us_ciks()
    if us_ciks:
        print(f"Loaded {len(us_ciks):,} U.S. CIKs for filtering", flush=True)
    else:
        print("No U.S. CIK filter loaded. Processing all CIKs.", flush=True)

    # Process facts parts
    print("\nProcessing companyfacts parts...", flush=True)
    process_start = now_ts()
    output_path, stats = process_facts_parts(us_ciks)
    process_elapsed = now_ts() - process_start

    print(f"\nProcessing completed in {fmt_elapsed(process_elapsed)}", flush=True)
    print(f"Total output rows: {stats['total_output_rows']:,}", flush=True)
    print(f"Observations processed: {stats['total_observations_processed']:,}", flush=True)

    # Write summary
    summary = {
        "output_file": str(output_path),
        "point_in_time_method": "strict_earliest_filing",
        "quarterly_forms": list(QUARTERLY_FORMS),
        "concepts_extracted": len(CONCEPT_MAP),
        "statistics": stats,
        "timing": {
            "processing_seconds": process_elapsed,
            "total_seconds": now_ts() - overall_start,
        },
    }

    with SUMMARY_FINAL.open("w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    total_elapsed = now_ts() - overall_start

    print("\n=== CORE FUNDAMENTALS COMPLETE ===", flush=True)
    print(f"Output: {output_path}", flush=True)
    print(f"Summary: {SUMMARY_FINAL}", flush=True)
    print(f"Total elapsed: {fmt_elapsed(total_elapsed)}", flush=True)


if __name__ == "__main__":
    main()

# python data/sec_core_fundamentals.py

### data\sec_data_cleaning_pipeline.py

In [ ]:
#!/usr/bin/env python3
"""
SEC Data Cleaning Pipeline

Cleans and standardizes the three core SEC processed files:
1. cik_ticker_map.csv - Exchange filtering, missing value removal
2. issuer_master.csv - US-only filtering, exchange filtering, column pruning, ticker filling
3. fundamentals_features.csv - Deduplication, null column removal, ticker/name filling, outlier capping

All operations are performed in order and outputs are saved as cleaned versions.
"""

from __future__ import annotations

import argparse
import csv
import json
import os
import time
from pathlib import Path
from typing import Any, Optional, Set

# ============================================================
# PATHS
# ============================================================

dataPath = Path(os.getenv("dataPathGlobal", "data"))

# Input files
CIK_TICKER_MAP_IN = dataPath / "sec_edgar" / "processed" / "issuer_master" / "cik_ticker_map.csv"
ISSUER_MASTER_IN = dataPath / "sec_edgar" / "processed" / "issuer_master" / "issuer_master.csv"
FUNDAMENTALS_IN = dataPath / "sec_edgar" / "processed" / "fundamentals" / "fundamentals_features.csv"

# Output files
OUT_DIR = dataPath / "sec_edgar" / "processed" / "cleaned"
CIK_TICKER_MAP_OUT = OUT_DIR / "cik_ticker_map_cleaned.csv"
ISSUER_MASTER_OUT = OUT_DIR / "issuer_master_cleaned.csv"
FUNDAMENTALS_OUT = OUT_DIR / "fundamentals_features_cleaned.csv"
SUMMARY_OUT = OUT_DIR / "cleaning_summary.json"

# ============================================================
# VALID EXCHANGES
# ============================================================

VALID_EXCHANGES = {"Nasdaq", "NYSE"}

# ============================================================
# OUTLIER CAPS (1000% = 10.0 in decimal form)
# ============================================================

RATIO_CAPS = {
    # Profitability margins (-1000% to 1000%)
    "gross_margin": (-10.0, 10.0),
    "operating_margin": (-10.0, 10.0),
    "net_margin": (-10.0, 10.0),
    "opex_to_revenue": (-10.0, 10.0),
    "cogs_to_revenue": (-10.0, 10.0),
    
    # Returns (-1000% to 1000%)
    "roa": (-10.0, 10.0),
    "roe": (-10.0, 10.0),
    
    # Leverage (0 to 100)
    "debt_to_equity": (-10.0, 100.0),
    "debt_to_assets": (0.0, 10.0),
    "current_ratio": (0.0, 100.0),
    "quick_ratio": (0.0, 100.0),
    "cash_ratio": (0.0, 100.0),
    
    # Efficiency (0 to 100)
    "asset_turnover": (0.0, 100.0),
    "ppe_turnover": (0.0, 100.0),
    
    # Cash flow ratios (-1000% to 1000%)
    "ocf_to_revenue": (-10.0, 10.0),
    "fcf_to_revenue": (-10.0, 10.0),
    "capex_to_revenue": (-10.0, 10.0),
    "fcf_to_net_income": (-100.0, 100.0),
    "ocf_to_net_income": (-100.0, 100.0),
    
    # Per share (no cap needed, but keep reasonable)
    "revenue_per_share": (-1e9, 1e9),
    "book_value_per_share": (-1e9, 1e9),
    "ocf_per_share": (-1e9, 1e9),
    "fcf_per_share": (-1e9, 1e9),
    
    # Quality
    "accruals_to_assets": (-10.0, 10.0),
    
    # Growth rates (-1000% to 1000%)
    "revenue_growth_yoy": (-10.0, 10.0),
    "revenue_growth_qoq": (-10.0, 10.0),
    "net_income_growth_yoy": (-10.0, 10.0),
    "net_income_growth_qoq": (-10.0, 10.0),
    "operating_income_growth_yoy": (-10.0, 10.0),
    "operating_income_growth_qoq": (-10.0, 10.0),
    "eps_basic_growth_yoy": (-10.0, 10.0),
    "eps_basic_growth_qoq": (-10.0, 10.0),
    "total_assets_growth_yoy": (-10.0, 10.0),
    "total_assets_growth_qoq": (-10.0, 10.0),
}

# Columns to drop from fundamentals (100% null or not needed)
FUNDAMENTALS_DROP_COLUMNS = [
    "inventory",
    "goodwill", 
    "intangible_assets",
    "short_term_debt",
]


# ============================================================
# HELPERS
# ============================================================

def now_ts() -> float:
    return time.time()


def fmt_elapsed(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.1f}s"
    minutes = seconds / 60.0
    if minutes < 60:
        return f"{minutes:.1f}m"
    hours = minutes / 60.0
    return f"{hours:.2f}h"


def safe_float(value: Any) -> Optional[float]:
    """Convert to float, return None if invalid/missing."""
    if value is None or value == "":
        return None
    try:
        return float(value)
    except (ValueError, TypeError):
        return None


def cap_value(value: Any, cap_range: tuple[float, float]) -> str:
    """Cap a numeric value to the specified range."""
    if value is None or value == "":
        return ""
    
    try:
        num = float(value)
        min_val, max_val = cap_range
        if num < min_val:
            num = min_val
        elif num > max_val:
            num = max_val
        return str(num)
    except (ValueError, TypeError):
        return ""


# ============================================================
# STEP 1: Clean cik_ticker_map.csv
# ============================================================

def clean_cik_ticker_map() -> tuple[int, int, dict]:
    """
    Clean cik_ticker_map.csv:
    - Remove rows with missing primary_exchange
    - Keep only Nasdaq and NYSE exchanges
    """
    print("\n" + "=" * 60)
    print("STEP 1: Cleaning cik_ticker_map.csv")
    print("=" * 60)
    
    if not CIK_TICKER_MAP_IN.exists():
        raise FileNotFoundError(f"Input file not found: {CIK_TICKER_MAP_IN}")
    
    rows_in = 0
    rows_out = 0
    missing_exchange = 0
    invalid_exchange = 0
    
    # Build CIK -> ticker mapping for later use
    cik_to_ticker: dict[str, str] = {}
    cik_to_name: dict[str, str] = {}
    
    with CIK_TICKER_MAP_IN.open("r", newline="", encoding="utf-8") as inf:
        reader = csv.DictReader(inf)
        fieldnames = reader.fieldnames
        
        # Write output
        OUT_DIR.mkdir(parents=True, exist_ok=True)
        tmp_out = CIK_TICKER_MAP_OUT.with_suffix(".csv.part")
        
        with tmp_out.open("w", newline="", encoding="utf-8") as outf:
            writer = csv.DictWriter(outf, fieldnames=fieldnames)
            writer.writeheader()
            
            for row in reader:
                rows_in += 1
                exchange = row.get("primary_exchange", "").strip()
                cik = row.get("cik", "").strip()
                ticker = row.get("primary_ticker", "").strip()
                entity_name = row.get("entity_name", "").strip()
                
                # Check for missing exchange
                if not exchange:
                    missing_exchange += 1
                    continue
                
                # Check for valid exchange
                if exchange not in VALID_EXCHANGES:
                    invalid_exchange += 1
                    continue
                
                # Keep this row
                writer.writerow(row)
                rows_out += 1
                
                # Store mapping
                if cik:
                    cik_to_ticker[cik] = ticker
                    cik_to_name[cik] = entity_name
        
        tmp_out.replace(CIK_TICKER_MAP_OUT)
    
    print(f"  Input rows: {rows_in:,}")
    print(f"  Output rows: {rows_out:,}")
    print(f"  Removed - missing exchange: {missing_exchange:,}")
    print(f"  Removed - invalid exchange: {invalid_exchange:,}")
    print(f"  Output: {CIK_TICKER_MAP_OUT}")
    
    stats = {
        "cik_ticker_map": {
            "rows_in": rows_in,
            "rows_out": rows_out,
            "missing_exchange_removed": missing_exchange,
            "invalid_exchange_removed": invalid_exchange,
        }
    }
    
    return rows_out, cik_to_ticker, cik_to_name, stats


# ============================================================
# STEP 2: Clean issuer_master.csv
# ============================================================

def clean_issuer_master(cik_to_ticker: dict[str, str]) -> tuple[int, dict]:
    """
    Clean issuer_master.csv:
    - Keep only is_us_issuer == 1, then drop this column
    - Keep only Nasdaq, NYSE, or missing primary_exchange
    - Drop columns: all_tickers, padded_cik, entity_type, state_of_incorporation_desc, state_of_incorporation
    - Keep business_city
    - Fill missing tickers using cik_to_ticker mapping
    """
    print("\n" + "=" * 60)
    print("STEP 2: Cleaning issuer_master.csv")
    print("=" * 60)
    
    if not ISSUER_MASTER_IN.exists():
        raise FileNotFoundError(f"Input file not found: {ISSUER_MASTER_IN}")
    
    rows_in = 0
    rows_out = 0
    non_us_removed = 0
    invalid_exchange_removed = 0
    tickers_filled = 0
    
    columns_to_drop = {
        "all_tickers", "padded_cik", "entity_type",
        "state_of_incorporation_desc", "state_of_incorporation", "business_state", "mailing_city", "mailing_state"
    }
    
    with ISSUER_MASTER_IN.open("r", newline="", encoding="utf-8") as inf:
        reader = csv.DictReader(inf)
        original_fieldnames = list(reader.fieldnames or [])
        
        # Determine output fieldnames
        output_fieldnames = [f for f in original_fieldnames if f not in columns_to_drop]
        if "is_us_issuer" in output_fieldnames:
            output_fieldnames.remove("is_us_issuer")
        
        tmp_out = ISSUER_MASTER_OUT.with_suffix(".csv.part")
        
        with tmp_out.open("w", newline="", encoding="utf-8") as outf:
            writer = csv.DictWriter(outf, fieldnames=output_fieldnames)
            writer.writeheader()
            
            for row in reader:
                rows_in += 1
                
                # Filter: keep only US issuers
                is_us = row.get("is_us_issuer", "").strip()
                if is_us != "1":
                    non_us_removed += 1
                    continue
                
                # Filter: keep only valid exchanges or missing
                exchange = row.get("primary_exchange", "").strip()
                if exchange and exchange not in VALID_EXCHANGES:
                    invalid_exchange_removed += 1
                    continue
                
                # Fill missing ticker if possible
                cik = row.get("cik", "").strip()
                ticker = row.get("primary_ticker", "").strip()
                if not ticker and cik and cik in cik_to_ticker:
                    row["primary_ticker"] = cik_to_ticker[cik]
                    tickers_filled += 1
                
                # Build output row (drop specified columns and is_us_issuer)
                out_row = {k: v for k, v in row.items() 
                          if k not in columns_to_drop and k != "is_us_issuer"}
                
                writer.writerow(out_row)
                rows_out += 1
        
        tmp_out.replace(ISSUER_MASTER_OUT)
    
    print(f"  Input rows: {rows_in:,}")
    print(f"  Output rows: {rows_out:,}")
    print(f"  Removed - non-US issuer: {non_us_removed:,}")
    print(f"  Removed - invalid exchange: {invalid_exchange_removed:,}")
    print(f"  Tickers filled from mapping: {tickers_filled:,}")
    print(f"  Columns dropped: {len(columns_to_drop) + 1} (including is_us_issuer)")
    print(f"  Output: {ISSUER_MASTER_OUT}")
    
    stats = {
        "issuer_master": {
            "rows_in": rows_in,
            "rows_out": rows_out,
            "non_us_removed": non_us_removed,
            "invalid_exchange_removed": invalid_exchange_removed,
            "tickers_filled": tickers_filled,
        }
    }
    
    return rows_out, stats


# ============================================================
# STEP 3: Clean fundamentals_features.csv
# ============================================================

def clean_fundamentals(cik_to_ticker: dict[str, str], cik_to_name: dict[str, str]) -> tuple[int, dict]:
    """
    Clean fundamentals_features.csv:
    - Deduplicate (keep first occurrence of each cik+fiscal_year+fiscal_period+filing_date)
    - Drop 100% null columns
    - Fill missing ticker and entity_name using cik_to_ticker and cik_to_name
    - Cap outlier values
    """
    print("\n" + "=" * 60)
    print("STEP 3: Cleaning fundamentals_features.csv")
    print("=" * 60)
    
    if not FUNDAMENTALS_IN.exists():
        raise FileNotFoundError(f"Input file not found: {FUNDAMENTALS_IN}")
    
    rows_in = 0
    rows_out = 0
    duplicates_removed = 0
    tickers_filled = 0
    names_filled = 0
    values_capped = 0
    
    # Track seen periods for deduplication
    seen_periods: Set[tuple] = set()
    
    with FUNDAMENTALS_IN.open("r", newline="", encoding="utf-8") as inf:
        reader = csv.DictReader(inf)
        original_fieldnames = list(reader.fieldnames or [])
        
        # Determine output fieldnames (exclude drop columns)
        output_fieldnames = [f for f in original_fieldnames if f not in FUNDAMENTALS_DROP_COLUMNS]
        
        tmp_out = FUNDAMENTALS_OUT.with_suffix(".csv.part")
        
        with tmp_out.open("w", newline="", encoding="utf-8") as outf:
            writer = csv.DictWriter(outf, fieldnames=output_fieldnames)
            writer.writeheader()
            
            for row in reader:
                rows_in += 1
                
                # Deduplication key
                cik = row.get("cik", "").strip()
                fiscal_year = row.get("fiscal_year", "").strip()
                fiscal_period = row.get("fiscal_period", "").strip()
                filing_date = row.get("filing_date", "").strip()
                
                dedup_key = (cik, fiscal_year, fiscal_period, filing_date)
                if dedup_key in seen_periods:
                    duplicates_removed += 1
                    continue
                seen_periods.add(dedup_key)
                
                # Fill missing ticker
                ticker = row.get("ticker", "").strip()
                if not ticker and cik and cik in cik_to_ticker:
                    row["ticker"] = cik_to_ticker[cik]
                    tickers_filled += 1
                
                # Fill missing entity_name
                entity_name = row.get("entity_name", "").strip()
                if not entity_name and cik and cik in cik_to_name:
                    row["entity_name"] = cik_to_name[cik]
                    names_filled += 1
                
                # Cap outlier values
                for col, cap_range in RATIO_CAPS.items():
                    if col in row:
                        original_val = row[col]
                        capped_val = cap_value(original_val, cap_range)
                        if capped_val != "" and original_val != "" and capped_val != original_val:
                            values_capped += 1
                        row[col] = capped_val
                
                # Build output row (exclude drop columns)
                out_row = {k: v for k, v in row.items() if k not in FUNDAMENTALS_DROP_COLUMNS}
                writer.writerow(out_row)
                rows_out += 1
                
                # Progress indicator
                if rows_in % 1000 == 0:
                    print(f"  Processed {rows_in:,} rows...", flush=True)
        
        tmp_out.replace(FUNDAMENTALS_OUT)
    
    print(f"  Input rows: {rows_in:,}")
    print(f"  Output rows: {rows_out:,}")
    print(f"  Duplicates removed: {duplicates_removed:,}")
    print(f"  Tickers filled: {tickers_filled:,}")
    print(f"  Entity names filled: {names_filled:,}")
    print(f"  Values capped: {values_capped:,}")
    print(f"  Columns dropped: {len(FUNDAMENTALS_DROP_COLUMNS)} (100% null columns)")
    print(f"  Output: {FUNDAMENTALS_OUT}")
    
    stats = {
        "fundamentals_features": {
            "rows_in": rows_in,
            "rows_out": rows_out,
            "duplicates_removed": duplicates_removed,
            "tickers_filled": tickers_filled,
            "entity_names_filled": names_filled,
            "values_capped": values_capped,
        }
    }
    
    return rows_out, stats


# ============================================================
# MAIN
# ============================================================

def main() -> None:
    parser = argparse.ArgumentParser(
        description="Clean and standardize SEC processed data files."
    )
    parser.add_argument(
        "--overwrite",
        action="store_true",
        help="Overwrite existing cleaned outputs.",
    )
    args = parser.parse_args()
    
    # Check if outputs already exist
    if not args.overwrite:
        existing = []
        if CIK_TICKER_MAP_OUT.exists():
            existing.append(str(CIK_TICKER_MAP_OUT))
        if ISSUER_MASTER_OUT.exists():
            existing.append(str(ISSUER_MASTER_OUT))
        if FUNDAMENTALS_OUT.exists():
            existing.append(str(FUNDAMENTALS_OUT))
        
        if existing:
            print("Some output files already exist:")
            for f in existing:
                print(f"  {f}")
            print("\nUse --overwrite to regenerate.")
            return
    
    overall_start = now_ts()
    
    print("=" * 60)
    print("SEC DATA CLEANING PIPELINE")
    print("=" * 60)
    
    all_stats = {}
    
    # Step 1: Clean cik_ticker_map
    _, cik_to_ticker, cik_to_name, stats1 = clean_cik_ticker_map()
    all_stats.update(stats1)
    
    # Step 2: Clean issuer_master
    _, stats2 = clean_issuer_master(cik_to_ticker)
    all_stats.update(stats2)
    
    # Step 3: Clean fundamentals
    _, stats3 = clean_fundamentals(cik_to_ticker, cik_to_name)
    all_stats.update(stats3)
    
    # Write summary
    summary = {
        "input_files": {
            "cik_ticker_map": str(CIK_TICKER_MAP_IN),
            "issuer_master": str(ISSUER_MASTER_IN),
            "fundamentals_features": str(FUNDAMENTALS_IN),
        },
        "output_files": {
            "cik_ticker_map": str(CIK_TICKER_MAP_OUT),
            "issuer_master": str(ISSUER_MASTER_OUT),
            "fundamentals_features": str(FUNDAMENTALS_OUT),
        },
        "valid_exchanges": list(VALID_EXCHANGES),
        "ratio_caps_applied": {k: list(v) for k, v in RATIO_CAPS.items()},
        "columns_dropped_from_fundamentals": FUNDAMENTALS_DROP_COLUMNS,
        "statistics": all_stats,
        "timing": {
            "total_seconds": now_ts() - overall_start,
        },
    }
    
    with SUMMARY_OUT.open("w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)
    
    total_elapsed = now_ts() - overall_start
    
    print("\n" + "=" * 60)
    print("CLEANING PIPELINE COMPLETE")
    print("=" * 60)
    print(f"\nOutput directory: {OUT_DIR}")
    print(f"Summary: {SUMMARY_OUT}")
    print(f"Total elapsed: {fmt_elapsed(total_elapsed)}")
    
    # Final summary
    print("\nFinal cleaned datasets:")
    print(f"  cik_ticker_map: {all_stats['cik_ticker_map']['rows_out']:,} rows")
    print(f"  issuer_master: {all_stats['issuer_master']['rows_out']:,} rows")
    print(f"  fundamentals_features: {all_stats['fundamentals_features']['rows_out']:,} rows")


if __name__ == "__main__":
    main()

# Run cleaning pipeline
# python data/sec_data_cleaning_pipeline.py
# Use --overwrite if needed later
# python data/sec_data_cleaning_pipeline.py --overwrite




### data\sec_fundamentals_features_3rdstep.py

In [ ]:
#!/usr/bin/env python3
"""
SEC Fundamentals Feature Engineering

Derives normalized ratios and growth metrics from point-in-time quarterly
fundamentals. All features are percentages or ratios, making them comparable
across companies of different sizes.

Input:  core_fundamentals_quarterly.csv
Output: fundamentals_features.csv (enriched with derived metrics)

Valuation metrics (P/E, P/B, P/S) are intentionally omitted until price data
is available from market data pipeline.
"""

from __future__ import annotations

import argparse
import csv
import json
import os
import time
from pathlib import Path
from typing import Any, Optional

# ============================================================
# PATHS
# ============================================================

dataPath = Path(os.getenv("dataPathGlobal", "data"))

INPUT_FILE = dataPath / "sec_edgar" / "processed" / "fundamentals" / "core_fundamentals_quarterly.csv"
OUTPUT_FILE = dataPath / "sec_edgar" / "processed" / "fundamentals" / "fundamentals_features.csv"
SUMMARY_FILE = dataPath / "sec_edgar" / "processed" / "fundamentals" / "fundamentals_features_summary.json"


# ============================================================
# HELPERS
# ============================================================

def now_ts() -> float:
    return time.time()


def fmt_elapsed(seconds: float) -> str:
    if seconds < 60:
        return f"{seconds:.1f}s"
    minutes = seconds / 60.0
    if minutes < 60:
        return f"{minutes:.1f}m"
    hours = minutes / 60.0
    return f"{hours:.2f}h"


def safe_float(value: Any) -> Optional[float]:
    """Convert to float, return None if invalid/missing."""
    if value is None or value == "":
        return None
    try:
        return float(value)
    except (ValueError, TypeError):
        return None


def safe_divide(numerator: Optional[float], denominator: Optional[float]) -> Optional[float]:
    """Divide safely, return None if division by zero or missing values."""
    if numerator is None or denominator is None:
        return None
    if denominator == 0:
        return None
    return numerator / denominator


def safe_growth(current: Optional[float], prior: Optional[float]) -> Optional[float]:
    """Calculate growth rate (current - prior) / abs(prior). Returns None if invalid."""
    if current is None or prior is None:
        return None
    if prior == 0:
        return None
    return (current - prior) / abs(prior)


def calculate_derived_features(row: dict) -> dict:
    """
    Calculate all derived ratios and percentages for a single period.
    These are normalized metrics comparable across companies.
    """
    # Extract raw values
    revenue = safe_float(row.get("revenue"))
    cost_of_revenue = safe_float(row.get("cost_of_revenue"))
    gross_profit = safe_float(row.get("gross_profit"))
    operating_expenses = safe_float(row.get("operating_expenses"))
    operating_income = safe_float(row.get("operating_income"))
    net_income = safe_float(row.get("net_income"))
    
    total_assets = safe_float(row.get("total_assets"))
    current_assets = safe_float(row.get("current_assets"))
    cash = safe_float(row.get("cash_and_equivalents"))
    inventory = safe_float(row.get("inventory"))
    ppe_net = safe_float(row.get("ppe_net"))
    
    total_liabilities = safe_float(row.get("total_liabilities"))
    current_liabilities = safe_float(row.get("current_liabilities"))
    long_term_debt = safe_float(row.get("long_term_debt"))
    short_term_debt = safe_float(row.get("short_term_debt"))
    
    shareholders_equity = safe_float(row.get("shareholders_equity"))
    
    operating_cash_flow = safe_float(row.get("operating_cash_flow"))
    capex = safe_float(row.get("capex"))
    free_cash_flow = safe_float(row.get("free_cash_flow"))
    
    shares_basic = safe_float(row.get("shares_basic"))
    eps_basic = safe_float(row.get("eps_basic"))

    features = {}

    # ============================================================
    # PROFITABILITY RATIOS (all percentages of revenue)
    # ============================================================
    features["gross_margin"] = safe_divide(gross_profit, revenue)
    features["operating_margin"] = safe_divide(operating_income, revenue)
    features["net_margin"] = safe_divide(net_income, revenue)
    
    # Operating efficiency
    features["opex_to_revenue"] = safe_divide(operating_expenses, revenue)
    features["cogs_to_revenue"] = safe_divide(cost_of_revenue, revenue)

    # ============================================================
    # RETURN RATIOS (efficiency of capital use)
    # ============================================================
    features["roa"] = safe_divide(net_income, total_assets)  # Return on Assets
    features["roe"] = safe_divide(net_income, shareholders_equity)  # Return on Equity
    
    # Alternative ROE using average equity would require prior period, skip for now

    # ============================================================
    # LEVERAGE & LIQUIDITY RATIOS
    # ============================================================
    features["debt_to_equity"] = safe_divide(long_term_debt, shareholders_equity)
    features["debt_to_assets"] = safe_divide(total_liabilities, total_assets)
    features["current_ratio"] = safe_divide(current_assets, current_liabilities)
    features["quick_ratio"] = safe_divide(
        (current_assets - inventory) if current_assets and inventory else current_assets,
        current_liabilities
    )
    features["cash_ratio"] = safe_divide(cash, current_liabilities)

    # ============================================================
    # EFFICIENCY RATIOS
    # ============================================================
    features["asset_turnover"] = safe_divide(revenue, total_assets)
    features["ppe_turnover"] = safe_divide(revenue, ppe_net)  # Fixed asset turnover

    # ============================================================
    # CASH FLOW METRICS
    # ============================================================
    features["ocf_to_revenue"] = safe_divide(operating_cash_flow, revenue)
    features["fcf_to_revenue"] = safe_divide(free_cash_flow, revenue)
    features["capex_to_revenue"] = safe_divide(capex, revenue)
    features["fcf_to_net_income"] = safe_divide(free_cash_flow, net_income)
    features["ocf_to_net_income"] = safe_divide(operating_cash_flow, net_income)

    # ============================================================
    # PER-SHARE METRICS (already normalized by shares)
    # ============================================================
    features["revenue_per_share"] = safe_divide(revenue, shares_basic)
    features["book_value_per_share"] = safe_divide(shareholders_equity, shares_basic)
    features["ocf_per_share"] = safe_divide(operating_cash_flow, shares_basic)
    features["fcf_per_share"] = safe_divide(free_cash_flow, shares_basic)

    # ============================================================
    # EARNINGS QUALITY
    # ============================================================
    features["accruals_to_assets"] = safe_divide(
        (net_income - operating_cash_flow) if net_income and operating_cash_flow else None,
        total_assets
    )

    return features


def process_fundamentals() -> tuple[list[dict], dict]:
    """
    Read core fundamentals, calculate derived features and growth rates.
    Returns enriched rows and statistics.
    """
    if not INPUT_FILE.exists():
        raise FileNotFoundError(f"Input file not found: {INPUT_FILE}")

    print(f"Reading: {INPUT_FILE}", flush=True)
    
    # Load all rows and group by CIK
    rows_by_cik: dict[str, list[dict]] = {}
    
    with INPUT_FILE.open("r", newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        fieldnames = list(reader.fieldnames or [])
        
        for row in reader:
            cik = row.get("cik", "").strip()
            if not cik:
                continue
            
            # Convert numeric strings to float for calculations
            if cik not in rows_by_cik:
                rows_by_cik[cik] = []
            rows_by_cik[cik].append(row)
    
    print(f"Loaded {sum(len(v) for v in rows_by_cik.values()):,} rows across {len(rows_by_cik):,} CIKs", flush=True)
    
    # Sort each CIK's rows chronologically
    for cik in rows_by_cik:
        rows_by_cik[cik].sort(key=lambda r: (
            r.get("fiscal_year", "0"),
            {"Q1": 1, "Q2": 2, "Q3": 3, "Q4": 4, "FY": 5}.get(r.get("fiscal_period", ""), 99)
        ))
    
    # Define new feature columns
    derived_feature_names = [
        "gross_margin", "operating_margin", "net_margin",
        "opex_to_revenue", "cogs_to_revenue",
        "roa", "roe",
        "debt_to_equity", "debt_to_assets",
        "current_ratio", "quick_ratio", "cash_ratio",
        "asset_turnover", "ppe_turnover",
        "ocf_to_revenue", "fcf_to_revenue", "capex_to_revenue",
        "fcf_to_net_income", "ocf_to_net_income",
        "revenue_per_share", "book_value_per_share",
        "ocf_per_share", "fcf_per_share",
        "accruals_to_assets",
    ]
    
    growth_feature_names = []
    base_metrics = ["revenue", "net_income", "operating_income", "eps_basic", "total_assets"]
    for metric in base_metrics:
        growth_feature_names.append(f"{metric}_growth_yoy")
        growth_feature_names.append(f"{metric}_growth_qoq")
    
    all_new_fields = derived_feature_names + growth_feature_names
    output_fieldnames = fieldnames + all_new_fields
    
    enriched_rows = []
    stats = {
        "total_rows": 0,
        "rows_with_growth": 0,
        "ciks_processed": len(rows_by_cik),
        "features_added": len(all_new_fields),
    }
    
    for cik, rows in rows_by_cik.items():
        for i, row in enumerate(rows):
            # Calculate derived features for current period
            features = calculate_derived_features(row)
            
            # Calculate growth rates if prior period exists
            if i > 0:
                prior_row = rows[i - 1]
                
                # YoY growth (same fiscal period, previous year)
                for metric in base_metrics:
                    current_val = safe_float(row.get(metric))
                    
                    # Find same period in prior year
                    yoy_val = None
                    for prior in rows[:i]:
                        if (prior.get("fiscal_period") == row.get("fiscal_period") and
                            int(prior.get("fiscal_year", 0)) == int(row.get("fiscal_year", 0)) - 1):
                            yoy_val = safe_float(prior.get(metric))
                            break
                    
                    features[f"{metric}_growth_yoy"] = safe_growth(current_val, yoy_val)
                    
                # QoQ growth (sequential periods)
                for metric in base_metrics:
                    current_val = safe_float(row.get(metric))
                    prior_val = safe_float(prior_row.get(metric))
                    features[f"{metric}_growth_qoq"] = safe_growth(current_val, prior_val)
                    
                stats["rows_with_growth"] += 1
            else:
                # First period for this CIK, no growth available
                for metric in base_metrics:
                    features[f"{metric}_growth_yoy"] = None
                    features[f"{metric}_growth_qoq"] = None
            
            # Merge features into row
            enriched_row = {**row}
            for key, value in features.items():
                enriched_row[key] = "" if value is None else str(value)
            
            enriched_rows.append(enriched_row)
            stats["total_rows"] += 1
    
    return enriched_rows, stats, output_fieldnames


def write_output(rows: list[dict], fieldnames: list[str]) -> None:
    """Write enriched data to CSV."""
    OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
    
    tmp_path = OUTPUT_FILE.with_suffix(".csv.part")
    with tmp_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    
    tmp_path.replace(OUTPUT_FILE)
    print(f"\nWritten: {OUTPUT_FILE}", flush=True)
    print(f"  Rows: {len(rows):,}", flush=True)
    print(f"  Columns: {len(fieldnames)} ({len(fieldnames) - 36} new features)", flush=True)


def write_summary(stats: dict, timing: dict) -> None:
    """Write summary JSON."""
    summary = {
        "input_file": str(INPUT_FILE),
        "output_file": str(OUTPUT_FILE),
        "statistics": stats,
        "feature_groups": {
            "profitability": ["gross_margin", "operating_margin", "net_margin", "opex_to_revenue", "cogs_to_revenue"],
            "returns": ["roa", "roe"],
            "leverage_liquidity": ["debt_to_equity", "debt_to_assets", "current_ratio", "quick_ratio", "cash_ratio"],
            "efficiency": ["asset_turnover", "ppe_turnover"],
            "cash_flow": ["ocf_to_revenue", "fcf_to_revenue", "capex_to_revenue", "fcf_to_net_income", "ocf_to_net_income"],
            "per_share": ["revenue_per_share", "book_value_per_share", "ocf_per_share", "fcf_per_share"],
            "earnings_quality": ["accruals_to_assets"],
            "growth": ["revenue_growth_yoy", "revenue_growth_qoq", "net_income_growth_yoy", "net_income_growth_qoq",
                      "operating_income_growth_yoy", "operating_income_growth_qoq", "eps_basic_growth_yoy",
                      "eps_basic_growth_qoq", "total_assets_growth_yoy", "total_assets_growth_qoq"],
        },
        "notes": [
            "All derived features are normalized ratios or percentages comparable across companies",
            "Growth rates calculated as (current - prior) / abs(prior)",
            "Valuation metrics (P/E, P/B, P/S) omitted - require price data",
            "YoY growth uses same fiscal period in prior year",
            "QoQ growth uses immediate prior period",
        ],
        "timing": timing,
    }
    
    with SUMMARY_FILE.open("w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)
    
    print(f"Summary: {SUMMARY_FILE}", flush=True)


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Derive normalized features from SEC core fundamentals."
    )
    parser.add_argument(
        "--overwrite",
        action="store_true",
        help="Overwrite existing output file.",
    )
    args = parser.parse_args()
    
    if not args.overwrite and OUTPUT_FILE.exists():
        print(f"Output already exists: {OUTPUT_FILE}")
        print("Use --overwrite to regenerate.")
        return
    
    overall_start = now_ts()
    
    print("=" * 60, flush=True)
    print("SEC FUNDAMENTALS FEATURE ENGINEERING", flush=True)
    print("=" * 60, flush=True)
    print("\nDeriving normalized ratios and growth metrics...", flush=True)
    
    process_start = now_ts()
    enriched_rows, stats, fieldnames = process_fundamentals()
    process_elapsed = now_ts() - process_start
    
    write_output(enriched_rows, fieldnames)
    
    timing = {
        "processing_seconds": process_elapsed,
        "total_seconds": now_ts() - overall_start,
    }
    
    write_summary(stats, timing)
    
    print("\n=== FEATURE ENGINEERING COMPLETE ===", flush=True)
    print(f"Rows processed: {stats['total_rows']:,}", flush=True)
    print(f"CIKs: {stats['ciks_processed']:,}", flush=True)
    print(f"Features added: {stats['features_added']}", flush=True)
    print(f"Rows with growth metrics: {stats['rows_with_growth']:,}", flush=True)
    print(f"Elapsed: {fmt_elapsed(timing['total_seconds'])}", flush=True)


if __name__ == "__main__":
    main()

# Run feature engineering
# python data/sec_fundamentals_features_3rdstep.py
# Use --overwrite if needed later
# python data/sec_fundamentals_features_3rdstep.py --overwrite